# AKE Lab 8 · Monitoring & Drift: Das Trägermodell vom laufenden Dienst zum Drift-Alarm — Starter-Notebook

**Angewandte KI & KI-Engineering · Block C – KI-Engineering & MLOps · Woche 8**
Bezug: Foliensatz *Monitoring, Drift & Lifecycle Management*, Abschnitte 2 (Drift-Arten), 3 (Drift erkennen),
4 (Monitoring & Observability) und 5 (Live am Trägermodell) · Lektüre: Huyen (2022), *Designing Machine Learning Systems*,
Kap. 8; Rabanser, Günnemann & Lipton (2019), *Failing Loudly*

> **Dieses Notebook ist das komplette Lab – es gibt kein separates Aufgabenblatt.** Jede Aufgabe enthält die Aufgabenstellung,
> die Formeln, die Erwartungswerte und die Leitfragen. Der Code ist ausführlich kommentiert: Lesen Sie die Kommentare
> (Zeilen, die mit `#` beginnen) – sie erklären Zeile für Zeile, was passiert.

**Die Aufgabe:** Seit Lab 7 läuft das Trägermodell als Dienst. Er antwortet auf jedes Bild mit einer Klasse und einer Zahl –
und meldet auch dann `200 OK`, wenn die Antwort falsch ist. Heute bauen Sie die Überwachung: Der Dienst bekommt ein
**Protokoll** (Inferenz-Logging), Sie schicken **zwei Bildströme** hindurch – die Validierungsbilder unverändert und
dieselben Bilder mit hellerem Licht, Unschärfe und JPEG-Kompression –, bauen daraus einen **Drift-Detektor** (PSI und
KS-Test mit Bonferroni-Korrektur), vergleichen ihn mit **Evidently** und legen eine **Schwelle** fest, ab der Alarm
geschlagen und nachtrainiert wird.

**Charakter:** Die Schritte wurden in der Vorlesung am Beamer gezeigt (*Live 1* und *Live 2*); Sie bearbeiten das Lab
**zu Hause** in Ihrem Tempo. **Ihre Zeit ≈ 100–120 Min.**, davon **≈ 1–2 Min. Rechenzeit** (Referenz 4 Kerne / 8 GB, keine
GPU, kein Docker).

## Organisatorisches

| | |
|---|---|
| **Termin** | Woche 8 (KW 48, 26.11.2026): Live-Demo in der Vorlesung, Bearbeitung zu Hause |
| **Bezug** | Foliensatz *Monitoring, Drift & Lifecycle Management*, Folien „Live 1: Zwei Bildströme durch den Dienst“, „Live 2: Vom Log zum Alarm“, „Lab 08 zu Hause“ und Anhang B (Code zum Nachschlagen) |
| **Material** | Ordner `lab08/` im Kursrepository: dieses Notebook, `vorlage/` (Dienst aus Lab 7 für den Notfall) und `vorberechnet/` (die Logs beider Ströme aus dem Referenzlauf); die Musterlösung folgt nach der Abgabe |
| **Umgebung** | Kursumgebung aktualisieren: `git pull`, danach `uv sync`. evidently 0.4.40, scipy 1.18.1 und torchvision 0.23.0 sind schon enthalten. **Kein Docker nötig** – der Dienst läuft mit `uvicorn`. |
| **Voraussetzung** | Ihr Übungs-Repo `traeger-pipeline/` (neben dem Kursordner) mit dem Dienst aus Lab 7 (`lab07/app.py`). Fehlt der Dienst: in Aufgabe 0 `OHNE_LAB7 = True` setzen. Fehlt das ganze Übungs-Repo: Das Notebook rechnet ab Aufgabe 3 mit den vorberechneten Logs. |

**Fahrplan**

| Aufgabe | In der Vorlesung gezeigt | Inhalt | Ihre Zeit | Rechenzeit |
|---|---|---|---|---|
| 0 · Setup | – | Pfade, Hilfsfunktionen, Prüfungen | 3 Min. | Sekunden |
| 1 · Dienst mit Protokoll | Live 1, Schritte 1 und 4 | `lab08/app.py` mit Forward-Hook und Log-Zeile (`CODE`), was ein Hook tut (`PRINZIP`), starten und das Log lesen (`CODE`) | 25 Min. | Sekunden |
| 2 · Zwei Bildströme | Live 1, Schritte 2, 3 und 5 | die drei Störungen von Hand (`PRINZIP`), `lab08/senden.py` (`CODE`), zwei Bilder – je zwei Antworten (`AUSFÜHREN`), Fenster aus dem Log laden (`AUSFÜHREN`) | 20 Min. | ≈ 20–60 s |
| 3 · Drift-Detektor selbst bauen | Live 2, Schritte 1–3 | PSI von Hand und als Funktion (`PRINZIP` + `CODE`), KS-Statistik von Hand (`PRINZIP` + `CODE`), 512 Tests und Bonferroni (`PRINZIP` + `CODE`), Gegenprobe (`CODE`) | 35 Min. | Sekunden |
| 4 · Evidently und Schwellwert | Live 2, Schritte 4 und 5 | Evidently-Bericht und seine Regel von Hand (`CODE` + `PRINZIP`), zweistufige Alarmregel (`CODE` + `PRINZIP`), Schwellwert begründen (`DENKEN`, Abgabe) | 25 Min. | Sekunden |
| Zusatz | – | Z1 schleichender Drift über sechs „Tage“ · Z2 Label Shift | – | ≈ 30–60 s |

**Kennzeichnung der Arbeitsarten** (steht am Anfang jeder Teilaufgabe):

| Marke | Was Sie tun |
|---|---|
| `CODE` | eine Lücke selbst füllen: die Zeile `raise NotImplementedError("TODO …")` durch Ihre Lösung ersetzen (der Kommentar darüber sagt, was hingehört) |
| `PRINZIP` | ausführen und lesen – bewusst einfacher Code (Schleifen, `print`), der zeigt, was eine Formel oder ein Werkzeug intern tut, und am Ende mit der Bibliothek vergleicht; oft mit einer kleinen Lücke |
| `AUSFÜHREN` | nur ausführen und die Ausgabe lesen |
| `DENKEN` | Interpretationsfrage in der Markdown-Zelle beantworten |

Wer bei einer Lücke nicht weiterkommt: Der Code steht auf den Folien in Anhang B. Die Aufgaben bauen aufeinander auf.

**Abgabe** (Weg und Termin: siehe Lab-Briefing): dieses Notebook **mit Ausgaben** und Ihren Antworten auf die
`DENKEN`-Fragen – vor allem **4.3: Schwellwert in fünf Sätzen begründen**. Die letzte Zelle packt zusätzlich
`lab08/app.py` und `lab08/senden.py` in ein ZIP.

## Lernziele

Nach dem Lab können Sie …

- **Verstehen:** erklären, warum ein Dienst ohne Protokoll nicht überwacht werden kann, und was ein Forward-Hook abgreift.
- **Anwenden:** einen Dienst um Inferenz-Logging erweitern und aus dem Log zwei Zeitfenster (Referenz, aktuell) laden.
- **Anwenden:** PSI und KS-Statistik von Hand und mit `scipy` berechnen – auf Konfidenzen und auf 512 Embedding-Dimensionen.
- **Analysieren:** begründen, warum viele Tests eine Korrektur brauchen (Bonferroni), und mit einer Gegenprobe prüfen, ob
  ein Detektor Fehlalarme gibt.
- **Analysieren:** die Regel eines fertigen Werkzeugs (Evidently) nachrechnen und mit der eigenen vergleichen.
- **Bewerten:** einen Schwellwert als Abwägung zwischen Fehlalarm und verpasstem Drift festlegen und begründen –
  und unterscheiden zwischen „Verteilung hat sich geändert“ und „Modell ist schlechter geworden“.

## Spielregeln und Hinweise

- **Sie arbeiten im Übungs-Repo.** Die beiden Dateien dieses Labs entstehen in `traeger-pipeline/lab08/` (neben dem
  Kursordner). Jupyter kann diesen Ordner im Dateibrowser nicht anzeigen – deshalb **schreibt eine Zelle die Datei**:
  `schreibe("lab08/app.py", "…Inhalt…")`. Sie ändern den Text in der Zelle und führen sie erneut aus.
- **Befehle laufen über `run("…")`.** Die Hilfsfunktion führt einen Befehl im Übungs-Repo aus und zeigt die Ausgabe so, wie
  Sie sie im Terminal sähen. Wer lieber tippt: Im Jupyter-Terminal (*File → New → Terminal*) mit `cd` in den Ordner
  `traeger-pipeline` wechseln – die Befehle sind dieselben.
- **Der Dienst läuft im Hintergrund.** `starte("dienst", "uvicorn lab08.app:app --port …", port)` startet ihn,
  `stoppe("dienst")` beendet ihn. Seine Ausgabe steht in `lab08/results/dienst.log` (im Kursordner).
- **Das Protokoll liegt im Übungs-Repo:** `traeger-pipeline/results/inferenz_log.jsonl` – eine Zeile je Anfrage.
- **Zellen mit `%%nur_mit_dienst` in der ersten Zeile** brauchen den Dienst bzw. das Modell. Ist beides nicht verfügbar
  (kein Übungs-Repo, Dienst startet nicht), überspringt das Notebook diese Zellen mit einem Hinweis und rechnet ab
  Aufgabe 2.4 mit den **vorberechneten Logs** aus `lab08/vorberechnet/`. Aufgaben 3 und 4 sind damit immer machbar.
- **Immer von oben nach unten.** Spätere Zellen brauchen die Dateien und Variablen der früheren. Eine Zelle zweimal
  auszuführen ist harmlos.
- **500 oder 501 Bilder?** Wer in Lab 6 (Aufgabe 1.4) die Kopie eines Bildes in die Validierungsliste aufgenommen hat,
  hat 501 Validierungsbilder, sonst 500. Beides ist richtig. Im Text steht 501 – das ist der Stand des Referenzlaufs.
- **Ihre Zahlen dürfen in der letzten Stelle abweichen.** Die Erwartungswerte stammen aus dem Referenzlauf (macOS, Apple
  Silicon). Auf anderen Rechnern rechnet PyTorch minimal anders; die Prüfungen lassen dafür Spielraum.

## Kurzhilfe: Notebook, Python und die Befehle dieses Labs

**Bedienung:** Zelle auswählen und mit **Shift + Enter** ausführen. Nach einem Fehler die Zelle korrigieren und erneut ausführen.

| Sie sehen im Code … | Bedeutung |
|---|---|
| `# Text` | Kommentar – wird nicht ausgeführt, erklärt nur |
| `run("python lab08/senden.py --strom drift")` | führt den Befehl im Übungs-Repo aus und zeigt die Ausgabe |
| `schreibe("lab08/app.py", r"""…""")` | legt eine Datei mit diesem Inhalt im Übungs-Repo an (überschreibt die alte) |
| `starte("dienst", "…", port)` · `stoppe("dienst")` | Programm im Hintergrund starten · beenden |
| `frage(URL, "/predict", {...})` | HTTP-Anfrage (POST mit JSON-Körper) schicken; Rückgabe `(Statuscode, Antwort)` |
| `d = {"a": 1}`, `d["a"]`, `list(d)` | *Dictionary* (Schlüssel → Wert) – eine Log-Zeile wird in Python zu einem Dictionary |
| `for x in liste:` | Schleife: Der eingerückte Block läuft für jedes Element |
| `f"{x:.3f}"` · `f"{x:.1%}"` | *f-String*: setzt `x` mit drei Nachkommastellen bzw. als Prozentzahl in den Text ein |
| `P.shape` | Form einer Matrix: `(501, 5)` = 501 Zeilen (Anfragen), 5 Spalten |
| `P.max(axis=1)` | größter Wert **je Zeile** → ein Wert je Anfrage (hier: die Konfidenz) |
| `E[:, 7]` · `E[:250]` | Spalte 7 aller Zeilen · die ersten 250 Zeilen |
| `(x < 0.6).mean()` | Anteil der Werte unter 0,6 (`True` zählt als 1, `False` als 0) |
| `np.log(x)` | natürlicher Logarithmus ln |
| `check("Name", lambda: ...)` | Prüfzelle: zeigt ✅, wenn die Bedingung hinter `lambda:` wahr ist, sonst ❌ mit Hinweis |
| `raise NotImplementedError("TODO …")` | Platzhalter in einer Lücke: Diese Zeile ersetzen Sie durch Ihre Lösung |

| Befehl | Was er tut |
|---|---|
| `uvicorn lab08.app:app [--port 8001]` | Dienst mit Protokoll starten: „im Paket `lab08` die Datei `app.py`, darin das Objekt `app`“ |
| `MODEL_URI=lab07/model uvicorn lab08.app:app` | derselbe Dienst, Modell aus dem Ordner `lab07/model` statt aus der Registry |
| `python lab08/senden.py --strom referenz` | die 501 Validierungsbilder unverändert an `/predict` schicken |
| `python lab08/senden.py --strom drift` | dieselben Bilder gestört schicken (Helligkeit, Unschärfe, JPEG) |
| `head -n 1 results/inferenz_log.jsonl` | erste Zeile des Protokolls ansehen (Windows PowerShell: `Get-Content results\inferenz_log.jsonl -Head 1`) |

**Begriffe, die heute vorkommen:** *Konfidenz* = größte der fünf Softmax-Wahrscheinlichkeiten einer Antwort ·
*Embedding* = die 512 Zahlen, die das Modell vor der letzten Schicht aus einem Bild gemacht hat · *Referenz* = das
Fenster, von dem wir wissen, dass das Modell darauf gut war (Validierungsbilder) · *aktuell* = das Fenster, das wir prüfen.

## Aufgabe 0 · Setup

`AUSFÜHREN` · Bibliotheken laden, Pfade festlegen, die Hilfsfunktionen bereitstellen und prüfen, ob alles da ist.
Die letzte Zeile sagt Ihnen, in welchem **Modus** Sie arbeiten:

- **„Dienst möglich“** – der Normalfall: Sie bearbeiten alle Aufgaben.
- **„vorberechnete Logs“** – Übungs-Repo oder Dienst fehlen. Lesen Sie Aufgaben 1 und 2 (die Zellen mit
  `%%nur_mit_dienst` überspringen sich selbst) und rechnen Sie ab 2.4 mit den Logs aus dem Referenzlauf.

**Lab 7 nicht abgeschlossen, aber Lab 6?** Setzen Sie in der ersten Zeile `OHNE_LAB7 = True` und führen Sie die Zelle
erneut aus: Sie legt den Dienst aus Lab 7 (`lab07/app.py`) und das Modell (`lab07/model/`) in Ihr Übungs-Repo.

In [ ]:
OHNE_LAB7 = False      # True: Dienst aus Lab 7 (lab07/app.py) und Modell (lab07/model/) hier ins Uebungs-Repo legen

# --- Bibliotheken -------------------------------------------------------------------------------------------
import os, re, sys, time, json, math, shutil, shlex, subprocess, types, warnings, logging    # Standardbibliothek
import base64, io, socket, signal, atexit, zipfile                                           # Standardbibliothek (Netz, Dateien)
import urllib.request, urllib.error                                                          # HTTP-Anfragen ohne Zusatzpaket
from pathlib import Path                         # Pfade zu Dateien und Ordnern, unabhaengig vom Betriebssystem
import numpy as np                               # Rechnen mit Vektoren und Matrizen
import pandas as pd                              # Tabellen (DataFrame)
import matplotlib.pyplot as plt                  # Diagramme und Bilder
from PIL import Image                            # Bilder oeffnen
import scipy
from scipy.stats import ks_2samp, binom          # KS-Test fuer zwei Stichproben; Binomialverteilung (Aufgabe 4.2)
import torch
import torchvision
import torchvision.transforms.v2.functional as F # Bildoperationen: Helligkeit, Unschaerfe, JPEG
warnings.filterwarnings("ignore")                # Hinweise von Bibliotheken ausblenden (lenken nur ab)
logging.getLogger("mlflow").setLevel(logging.ERROR)
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
import mlflow, mlflow.pytorch                    # nur zum Laden des Modells (wie in Lab 7)
NAVY, ORANGE, GRAU = "#1F3864", "#E87722", "#8A8A8A"          # Farben wie auf den Folien
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
np.set_printoptions(precision=3, suppress=True)

# --- Pruefhelfer --------------------------------------------------------------------------------------------
# check("Name", Bedingung, "Hinweis") gibt ✅ aus, wenn die Bedingung wahr ist, sonst ❌ mit Hinweis.
# Die Bedingung steht meist als `lambda: ...` da: eine kleine Funktion, die erst hier ausgewertet wird.
CHECKS = {}
def check(name, ok, hint=""):
    try:
        ok = bool(ok() if callable(ok) else ok); msg = hint
    except NotImplementedError as e:             # die Luecke ist noch nicht gefuellt
        ok, msg = False, f"{e} ist noch offen"
    except Exception as e:                       # irgendein anderer Fehler
        ok, msg = False, f"{type(e).__name__}: {e}" + (f" | {hint}" if hint else "")
    CHECKS[name] = ok
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {msg}")

def nah(a, b, tol):                              # |a - b| <= tol  (fuer Kontrollwerte mit Rundung)
    return abs(float(a) - float(b)) <= tol

# --- Laufzeitmessung je Aufgabe (Tabelle am Ende; fuer das Lab nicht wichtig) ---------------------------------
RUNTIME, _teil = {}, {"name": "Aufgabe 0", "t0": None}
def abschnitt(name): _teil["name"] = name
def _ake_pre(*a): _teil["t0"] = time.perf_counter()
def _ake_post(*a):
    if _teil["t0"] is not None:
        RUNTIME[_teil["name"]] = RUNTIME.get(_teil["name"], 0.0) + time.perf_counter() - _teil["t0"]
try:
    _ip = get_ipython()
    for ev, fn in (("pre_run_cell", _ake_pre), ("post_run_cell", _ake_post)):
        for old in [f for f in _ip.events.callbacks[ev] if getattr(f, "__name__", "") == fn.__name__]:
            _ip.events.unregister(ev, old)
        _ip.events.register(ev, fn)
except NameError:
    pass

# --- Pfade ----------------------------------------------------------------------------------------------------
# REPO_ROOT = Kursordner (erster Elternordner mit pyproject.toml). Das Uebungs-Repo ARBEIT (aus Lab 6/7) liegt DANEBEN.
# Anderer Ort: Umgebungsvariable AKE_LAB08_ARBEIT setzen oder den Pfad hier eintragen.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Kein Kursrepository gefunden. Bitte Jupyter im Ordner des Repositorys starten und lab08/ oeffnen.")
DATA         = REPO_ROOT / "data"
VORLAGE      = REPO_ROOT / "lab08" / "vorlage"                # lab07/app.py fuer den Notfallweg OHNE_LAB7
VORBERECHNET = REPO_ROOT / "lab08" / "vorberechnet"           # Logs beider Stroeme aus dem Referenzlauf (.npz)
ARBEIT  = Path(os.environ.get("AKE_LAB08_ARBEIT", os.environ.get("AKE_LAB07_ARBEIT", REPO_ROOT.parent / "traeger-pipeline"))).resolve()
RESULTS = Path("results").resolve(); RESULTS.mkdir(exist_ok=True)   # lab08/results im Kursordner: Bericht, Bilder, dienst.log
LOGDATEI = ARBEIT / "results" / "inferenz_log.jsonl"          # das Protokoll des Dienstes (im Uebungs-Repo)

# --- run(): einen Befehl im Uebungs-Repo ausfuehren ---------------------------------------------------------------
# Was die Funktion tut:
#   1. Den Befehl in seine Teile zerlegen ("python lab08/senden.py --strom drift" -> ["python", "lab08/senden.py", ...]).
#   2. "python" und "uvicorn" durch "<diese Python-Umgebung> [-m ...]" ersetzen - so laeuft garantiert das Programm der
#      Kursumgebung, auch wenn im System ein anderes Python installiert ist.
#   3. Den Befehl im Ordner ARBEIT starten und jede Ausgabezeile sofort anzeigen (wie im Terminal).
# umgebung={...} setzt zusaetzliche Umgebungsvariablen nur fuer diesen Befehl. Rueckgabe: Objekt mit .ok, .ausgabe, .sekunden.
LAUFZEITEN = {}                                               # Laufzeit je Befehl (Tabelle am Ende)
MODULE = {"uvicorn": "uvicorn", "pytest": "pytest", "mlflow": "mlflow"}

def _teile(befehl):
    teile = shlex.split(befehl) if isinstance(befehl, str) else [str(t) for t in befehl]
    if teile[0] == "python":
        teile = [sys.executable] + teile[1:]
    elif teile[0] in MODULE:
        teile = [sys.executable, "-m", MODULE[teile[0]]] + teile[1:]
    return teile

def _env(umgebung=None):
    env = dict(os.environ, PATH=str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", ""),
               PYTHONUNBUFFERED="1", PYTHONWARNINGS="ignore", PYTHONIOENCODING="utf-8",
               MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR="false")
    env.pop("MLFLOW_TRACKING_URI", None)          # der Dienst soll ./mlruns im Uebungs-Repo benutzen (Standard von MLflow)
    env.update({k: str(v) for k, v in (umgebung or {}).items()})
    return env

def run(befehl, erwarte_fehler=False, zeigen=True, cwd=None, umgebung=None):
    teile = _teile(befehl)
    text = befehl if isinstance(befehl, str) else " ".join(str(t) for t in befehl)
    if umgebung:
        text = " ".join(f"{k}={v}" for k, v in umgebung.items()) + " " + text
    if zeigen:
        print(f"$ {text}", flush=True)
    t0 = time.perf_counter()
    try:
        p = subprocess.Popen(teile, cwd=cwd or ARBEIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, encoding="utf-8", errors="replace", env=_env(umgebung))
    except FileNotFoundError:
        print(f"❌ Programm '{teile[0]}' nicht gefunden."); return types.SimpleNamespace(ok=False, ausgabe="", sekunden=0)
    zeilen = []
    for zeile in p.stdout:                                  # Zeile fuer Zeile, waehrend der Befehl laeuft
        zeile = zeile.rstrip()
        if not zeile and (not zeilen or not zeilen[-1]):
            continue                                        # doppelte Leerzeilen weglassen
        zeilen.append(zeile)
        if zeigen:
            print("   " + zeile, flush=True)
    p.wait()
    erg = types.SimpleNamespace(ok=(p.returncode == 0), ausgabe="\n".join(zeilen), sekunden=time.perf_counter() - t0)
    LAUFZEITEN[f"{len(LAUFZEITEN) + 1:3d}: {text}"[:70]] = erg.sekunden
    if not erg.ok and not erwarte_fehler:
        raise RuntimeError(f"Befehl fehlgeschlagen (Rueckgabewert {p.returncode}): {text} – Ausgabe oben lesen.")
    return erg

# --- schreibe(): eine Datei im Uebungs-Repo anlegen ------------------------------------------------------------------
# Jupyter erreicht den Ordner traeger-pipeline/ im Dateibrowser nicht - deshalb ist die Notebook-Zelle unser Editor.
OFFEN = re.compile(r'raise NotImplementedError\("(TODO [^"]+)"\)')    # eine noch nicht gefuellte Luecke in der Datei
def schreibe(relpfad, text, basis=None):
    ziel = (basis or ARBEIT) / relpfad
    ziel.parent.mkdir(parents=True, exist_ok=True)
    text = text.strip("\n") + "\n"
    ziel.write_text(text, encoding="utf-8", newline="\n")
    print(f"geschrieben: {relpfad} ({len(text.splitlines())} Zeilen)")
    luecke = OFFEN.search(text)
    if luecke:                                               # Luecke noch offen -> hier anhalten statt spaeter raetseln
        raise NotImplementedError(f"{luecke.group(1)} in {relpfad}")
    if relpfad.endswith(".py"):
        compile(text, relpfad, "exec")                       # Tippfehler/Einrueckung sofort melden (SyntaxError)
    return ziel

# --- starte() / stoppe(): Programme im Hintergrund (uvicorn) ----------------------------------------------------------
HINTERGRUND = globals().get("HINTERGRUND", {})                # Name -> (Prozess, Logdatei, Pfad der Logdatei)
STARTZEIT = {}                                                # Name -> Sekunden bis zur ersten Antwort
OHNE_PROXY = urllib.request.build_opener(urllib.request.ProxyHandler({}))   # Anfragen an 127.0.0.1 nie ueber einen Proxy

def port_frei(port):
    """True, wenn auf diesem Port noch kein Programm lauscht."""
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) != 0        # Verbindung scheitert -> Port ist frei

def freier_port(*kandidaten):
    """Ersten freien Port aus der Liste liefern."""
    for port in kandidaten:
        if port_frei(port):
            return port
    raise RuntimeError(f"Alle Ports belegt: {kandidaten} – laufende Dienste mit stoppe() beenden.")

def stoppe(name=None):
    """Ein Hintergrundprogramm beenden (ohne Namen: alle)."""
    for n in ([name] if name else list(HINTERGRUND)):
        if n not in HINTERGRUND:
            continue
        p, f, log = HINTERGRUND.pop(n)
        if p.poll() is None:
            if os.name == "nt":                              # Windows: Prozess samt Kindprozessen beenden
                subprocess.run(["taskkill", "/F", "/T", "/PID", str(p.pid)], capture_output=True)
            else:                                            # macOS/Linux: die ganze Prozessgruppe beenden
                try: os.killpg(p.pid, signal.SIGTERM)
                except ProcessLookupError: pass
            try: p.wait(10)
            except subprocess.TimeoutExpired: p.kill()
        f.close()
        print(f"⏹ {n} gestoppt")
atexit.register(stoppe)                                       # beim Beenden des Kernels aufraeumen

def warte_auf(url, sekunden=60, prozess=None):
    """Warten, bis unter url eine Antwort kommt. Rueckgabe: Wartezeit in Sekunden (oder None)."""
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < sekunden:
        if prozess is not None and prozess.poll() is not None:
            return None                                      # Programm hat sich beendet
        try:
            OHNE_PROXY.open(url, timeout=2).read(); return time.perf_counter() - t0
        except urllib.error.HTTPError:
            return time.perf_counter() - t0                  # Server antwortet (wenn auch mit Fehlercode)
        except Exception:
            time.sleep(0.4)
    return None

def starte(name, befehl, port, pfad="/", umgebung=None, sekunden=90):
    """Befehl im Uebungs-Repo im Hintergrund starten und warten, bis http://127.0.0.1:<port><pfad> antwortet."""
    stoppe(name)
    log = RESULTS / f"{name}.log"
    f = open(log, "w", encoding="utf-8")
    extra = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {"start_new_session": True}
    text = (" ".join(f"{k}={v}" for k, v in (umgebung or {}).items()) + " " + befehl).strip()
    print(f"$ {text}   (im Hintergrund)", flush=True)
    p = subprocess.Popen(_teile(befehl), cwd=ARBEIT, stdout=f, stderr=subprocess.STDOUT, env=_env(umgebung), **extra)
    HINTERGRUND[name] = (p, f, log)
    url = f"http://127.0.0.1:{port}"
    gewartet = warte_auf(url + pfad, sekunden, p)
    if gewartet is None:
        f.flush(); ende = "\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-15:])
        stoppe(name)
        raise RuntimeError(f"{name} ist nicht gestartet. Ende von {log.name}:\n{ende}\n"
                           "Laesst sich das nicht beheben: DIENST_MOEGLICH = False setzen und bei 2.4 weitermachen.")
    STARTZEIT[name] = gewartet
    print(f"✅ {name} laeuft nach {gewartet:.1f} s: {url}{pfad}   (Protokoll: results/{log.name}, beenden: stoppe(\"{name}\"))")
    return url

# --- frage(): eine HTTP-Anfrage schicken ----------------------------------------------------------------------------
def kurz(obj, n=60):
    """Lange Texte in einer Antwort/Anfrage fuer die Anzeige kuerzen."""
    if isinstance(obj, dict): return {k: kurz(v, n) for k, v in obj.items()}
    if isinstance(obj, list): return [kurz(v, n) for v in obj]
    if isinstance(obj, str) and len(obj) > n: return obj[:24] + f"… ({len(obj)} Zeichen)"
    return obj

def frage(url, pfad, daten=None, zeigen=True):
    """GET (ohne daten) oder POST mit JSON-Koerper (daten = Dictionary). Rueckgabe: (Statuscode, Antwort als Dictionary)."""
    methode = "GET" if daten is None else "POST"
    koerper = None if daten is None else json.dumps(daten).encode("utf-8")       # Dictionary -> JSON-Text -> Bytes
    req = urllib.request.Request(url + pfad, data=koerper, method=methode, headers={"Content-Type": "application/json"})
    t0 = time.perf_counter()
    try:
        with OHNE_PROXY.open(req, timeout=60) as r:
            status, text = r.status, r.read().decode("utf-8")
    except urllib.error.HTTPError as e:                      # Statuscodes ab 400 kommen als Ausnahme
        status, text = e.code, e.read().decode("utf-8")
    ms = (time.perf_counter() - t0) * 1000
    try: antwort = json.loads(text)
    except json.JSONDecodeError: antwort = {"text": text}
    if zeigen:
        print(f"{methode} {pfad}" + (f"  {json.dumps(kurz(daten), ensure_ascii=False)}" if daten is not None else ""))
        print(f"   -> {status} in {ms:.0f} ms: {json.dumps(kurz(antwort, 200), ensure_ascii=False)}")
    return status, antwort

def b64_png(bild):
    """PIL-Bild verlustfrei als PNG speichern und als Base64-Text zurueckgeben (so erwartet es /predict)."""
    puffer = io.BytesIO(); bild.save(puffer, format="PNG")
    return base64.b64encode(puffer.getvalue()).decode()

# --- Was ist da? ---------------------------------------------------------------------------------------------------
def modell_ort():
    """Wo liegt das Modell fuer den Dienst? Ordner lab07/model (Lab 7, Aufgabe 3.1) oder die Registry (Lab 7, Aufgabe 1.3)."""
    if (ARBEIT / "lab07" / "model" / "MLmodel").exists():
        return "lab07/model"
    if (ARBEIT / "mlruns" / "models" / "traeger").exists():
        return "models:/traeger@champion"
    return None

def lab6_stand_da():
    return all((ARBEIT / f).exists() for f in ("lab02/src/traeger.py", "models/traeger_v1.yaml", "data/imagenette5/val.txt"))

def lab7_stand_da():
    app = ARBEIT / "lab07" / "app.py"
    return lab6_stand_da() and app.exists() and "def vorhersage" in app.read_text(encoding="utf-8") and modell_ort() is not None

# --- Notfall: Dienst aus Lab 7 hier nachlegen -------------------------------------------------------------------------
if OHNE_LAB7 and not lab7_stand_da():
    if not (lab6_stand_da() and (ARBEIT / "models" / "traeger_v1.pt").exists()):
        print(f"ℹ️  OHNE_LAB7: In {ARBEIT} fehlt auch der Stand aus Lab 6 (models/traeger_v1.pt) – weiter mit den vorberechneten Logs.")
    else:
        if not (ARBEIT / "lab07" / "app.py").exists() or "def vorhersage" not in (ARBEIT / "lab07" / "app.py").read_text(encoding="utf-8"):
            (ARBEIT / "lab07").mkdir(exist_ok=True)
            shutil.copy2(VORLAGE / "lab07" / "app.py", ARBEIT / "lab07" / "app.py")
            print("angelegt: lab07/app.py (Dienst aus Lab 7)")
        if modell_ort() is None:
            sys.path.insert(0, str(ARBEIT))
            from lab02.src.traeger import load_traeger
            _m, _, _ = load_traeger(ARBEIT / "models" / "traeger_v1.yaml")     # Modell aus der Pipeline von Lab 6
            mlflow.pytorch.save_model(_m, str(ARBEIT / "lab07" / "model"))     # im MLflow-Format ablegen (wie Lab 7, Aufgabe 3.1)
            print("angelegt: lab07/model/ (Modell im MLflow-Format)")

DIENST_MOEGLICH = lab7_stand_da()                 # von Hand auf False setzen = mit den vorberechneten Logs arbeiten
MODEL_URI = modell_ort()
BILDER = (ARBEIT if lab6_stand_da() else DATA.parent) / "data" / "imagenette5"           # Validierungsbilder
KLASSEN = [k["name"] for k in json.loads((BILDER / "classes.json").read_text(encoding="utf-8"))]
VAL = [z.split("\t") for z in (BILDER / "val.txt").read_text().strip().splitlines()]      # je Zeile: Pfad <TAB> Klassenindex
BILD_KIRCHE = BILDER / next(pfad for pfad, label in VAL if int(label) == KLASSEN.index("Kirche"))   # erstes Kirchenbild

# --- Modell auch hier im Notebook laden (fuer die PRINZIP-Zellen und den Zusatz) ------------------------------------
modell = tf = None
if DIENST_MOEGLICH:
    if str(ARBEIT) not in sys.path:
        sys.path.insert(0, str(ARBEIT))                       # damit "from lab02.src.traeger import ..." das Uebungs-Repo findet
    from lab02.src.traeger import make_transform
    mlflow.set_tracking_uri((ARBEIT / "mlruns").as_uri())
    modell = mlflow.pytorch.load_model(str(ARBEIT / MODEL_URI) if MODEL_URI == "lab07/model" else MODEL_URI).eval()
    tf = make_transform(train=False)                          # dieselbe Vorverarbeitung wie im Dienst
    zeile = "results/"                                        # das Protokoll gehoert nicht ins Git-Repository
    gi = ARBEIT / ".gitignore"
    if gi.exists() and zeile not in gi.read_text(encoding="utf-8").splitlines():
        gi.write_text(gi.read_text(encoding="utf-8").rstrip("\n") + "\n" + zeile + "\n", encoding="utf-8")
        print("ergaenzt: .gitignore im Uebungs-Repo um die Zeile results/")

# --- %%nur_mit_dienst: Zellen, die Dienst oder Modell brauchen ------------------------------------------------------
# Steht %%nur_mit_dienst in der ersten Zeile einer Zelle, laeuft der Rest nur, wenn DIENST_MOEGLICH wahr ist.
def nur_mit_dienst(zeile, zelle):
    if not DIENST_MOEGLICH:
        print("⏭ Zelle uebersprungen (kein Dienst/Modell verfuegbar) – lesen Sie den Code und machen Sie bei 2.4 mit den vorberechneten Logs weiter.")
        return
    ip = get_ipython()
    try: name = ip.compile.cache(zelle)                       # Zelle registrieren, damit Fehlermeldungen die Codezeile zeigen
    except Exception: name = "<zelle>"
    exec(compile(zelle, name, "exec"), ip.user_ns)
try:
    get_ipython().register_magic_function(nur_mit_dienst, magic_kind="cell")
except NameError:
    pass

# --- Pruefungen -----------------------------------------------------------------------------------------------
import fastapi, uvicorn, httpx, evidently
print(f"Kursordner:   {REPO_ROOT}\nUebungs-Repo: {ARBEIT}")
print(f"evidently {evidently.__version__} | scipy {scipy.__version__} | torchvision {torchvision.__version__} | torch {torch.__version__} | FastAPI {fastapi.__version__}")
check("0 evidently 0.4.x in der Kursumgebung", lambda: evidently.__version__.startswith("0.4."), "im Kursordner: git pull, dann uv sync; danach Jupyter neu starten")
check("0 Validierungsbilder da (500 oder 501 Stueck)", lambda: len(VAL) in (500, 501) and BILD_KIRCHE.exists(), "git pull im Kursordner")
check("0 Vorberechnete Logs da (lab08/vorberechnet)", lambda: all((VORBERECHNET / f"log_{s}.npz").exists() for s in ("referenz", "drift")), "git pull im Kursordner")
if DIENST_MOEGLICH:
    print(f"✅ Modus: Dienst moeglich – Modell: {MODEL_URI} · alle Aufgaben bearbeiten")
else:
    print(f"ℹ️  Modus: vorberechnete Logs – in {ARBEIT} fehlt " +
          ("das Uebungs-Repo bzw. der Stand aus Lab 6" if not lab6_stand_da() else "der Dienst aus Lab 7 (oben OHNE_LAB7 = True setzen)") +
          ".\n    Aufgaben 1 und 2 lesen, ab 2.4 rechnen.")

## Aufgabe 1 · Dienst mit Protokoll

**In der Vorlesung: Live 1, Schritte 1 und 4 · ≈ 25 Min.** · Folien „Was der Dienst heute verrät“ und Anhang B
„Code: Inferenz-Logging in lab08/app.py“

Der Dienst aus Lab 7 antwortet mit `{"klasse", "p", "modell"}` – `p` ist nur die **größte** der fünf Wahrscheinlichkeiten.
Was das Modell sonst noch „gedacht“ hat, verlässt den Dienst nicht und ist nach der Antwort verloren. Für die Überwachung
brauchen wir je Anfrage eine **Log-Zeile** mit

- dem Zeitpunkt,
- allen fünf Wahrscheinlichkeiten (Softmax) und
- dem **Embedding**: den 512 Zahlen, die ResNet-18 vor der letzten Schicht aus dem Bild gemacht hat.

Am Ende läuft `uvicorn lab08.app:app`, antwortet genau wie bisher – und schreibt nebenbei `results/inferenz_log.jsonl`.

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 1.1 | `lab08/app.py` vervollständigen – und was ein Forward-Hook tut | `CODE` + `PRINZIP` | 15 Min. |
| 1.2 | Starten, ein Bild schicken, die Log-Zeile lesen | `CODE` | 10 Min. |

### 1.1 · `lab08/app.py` vervollständigen

`CODE` (2 Lücken) · Die neue Datei **verändert `lab07/app.py` nicht**. Sie importiert daraus Schema, Modell,
Vorverarbeitung und Klassennamen (das Modell wird dadurch genau einmal geladen) und ergänzt zwei Dinge:

- **1.1a – der Forward-Hook.** Ein *Hook* ist eine Funktion, die PyTorch nach einer bestimmten Schicht aufruft – mit der
  Ausgabe dieser Schicht. Unsere Funktion `merke` legt die Ausgabe der Schicht `avgpool` (das Embedding) in `emb["v"]` ab.
  Es fehlt die Zeile, die sie anmeldet: `model.avgpool.register_forward_hook(merke)`.
- **1.1b – die Log-Zeile.** Ein Dictionary mit vier Schlüsseln:
  `"t"`: `time.time()` (Sekunden seit 1970) · `"klasse"`: `classes[k]` · `"p"`: die fünf Wahrscheinlichkeiten als Liste,
  auf 6 Stellen gerundet: `[round(v, 6) for v in p.tolist()]` · `"emb"`: die 512 Werte des Embeddings, auf 4 Stellen
  gerundet: `[round(v, 4) for v in emb["v"][0].tolist()]`.

Die Antwort an den Client bleibt **dieselbe wie in Lab 7** – wer den Dienst benutzt, merkt vom Protokoll nichts.

In [ ]:
%%nur_mit_dienst
abschnitt("Aufgabe 1")
schreibe("lab08/app.py", r"""
# lab08/app.py - der Dienst des Traegermodells MIT Protokoll (Inferenz-Logging)
# Starten (aus dem Hauptordner des Repos):  uvicorn lab08.app:app
import base64, io, json, os, time

import torch
from fastapi import FastAPI, HTTPException
from PIL import Image

# Schema, Modell, Vorverarbeitung und Klassennamen kommen aus dem Dienst von Lab 7 - importiert, nicht kopiert.
# Beim Import laeuft lab07/app.py einmal durch: Das Modell wird also genau einmal geladen, wie bisher.
from lab07.app import Anfrage, Antwort, model, tf, classes, MODEL_URI

# ---- laeuft EINMAL beim Start -------------------------------------------------------------------------------
LOG = os.getenv("LOG", "results/inferenz_log.jsonl")      # Pfad des Protokolls; ueber die Umgebungsvariable LOG aenderbar
os.makedirs(os.path.dirname(LOG) or ".", exist_ok=True)   # Ordner results/ anlegen, falls er fehlt

emb = {}                                                  # hier legt der Hook das Embedding der letzten Anfrage ab
def merke(schicht, eingabe, ausgabe):                     # PyTorch ruft diese Funktion nach jedem Durchlauf der Schicht auf
    emb["v"] = ausgabe.flatten(1)                         # Ausgabe von avgpool: Form (1, 512, 1, 1) -> (1, 512)

# TODO 1.1a: den Hook an der Schicht avgpool anmelden: model.avgpool.register_forward_hook(merke)
raise NotImplementedError("TODO 1.1a")

app = FastAPI(title="Trägermodell-Service mit Protokoll")


# ---- Endpunkte ---------------------------------------------------------------------------------------------------
@app.get("/health")
def health():
    return {"status": "ok", "modell": MODEL_URI}          # wie in Lab 7


@app.post("/predict", response_model=Antwort)
def vorhersage(a: Anfrage):
    try:
        bild = Image.open(io.BytesIO(base64.b64decode(a.image_b64))).convert("RGB")
    except Exception:
        raise HTTPException(422, "image_b64 ist kein lesbares Bild")
    with torch.no_grad():                                 # Vorwaertsdurchlauf ohne Gradienten; dabei laeuft der Hook
        logits = model(tf(bild).unsqueeze(0))
    p = torch.softmax(logits, dim=1)[0]                   # fuenf Wahrscheinlichkeiten, Summe 1
    k = int(p.argmax())                                   # Index der groessten Wahrscheinlichkeit
    # TODO 1.1b: zeile = {...} mit den vier Schluesseln "t", "klasse", "p", "emb" (siehe Aufgabentext)
    raise NotImplementedError("TODO 1.1b")
    with open(LOG, "a", encoding="utf-8") as f:           # "a" = anhaengen: eine Zeile je Anfrage (Format JSON Lines)
        f.write(json.dumps(zeile) + "\n")
    return Antwort(klasse=classes[k], p=round(float(p[k]), 4), modell=MODEL_URI)     # Antwort wie in Lab 7
""")
check("1.1 app.py geschrieben: Hook angemeldet, Log-Zeile mit vier Schluesseln",
      lambda: all(s in (ARBEIT / "lab08/app.py").read_text(encoding="utf-8") for s in ("register_forward_hook(merke)", '"emb"', '"klasse"', '"t"')))

#### 1.1 `PRINZIP` · Was tut ein Forward-Hook?

`PRINZIP` + `CODE` (1 Zeile) · Ein neuronales Netz ist eine Kette von Schichten: Die Ausgabe der einen ist die Eingabe
der nächsten. Normalerweise sehen wir nur, was am Ende herauskommt. Ein Hook ist ein **Zuschauer an einer Zwischenstation**:
Er merkt sich, was dort vorbeikommt, und ändert nichts.

**Teil A – ein Mini-Netz aus drei Rechenschritten**, ganz ohne PyTorch.
**Teil B – das Trägermodell Schicht für Schicht von Hand:** Wir schicken das Kirchenbild durch die zehn Bausteine von
ResNet-18 und halten vor der letzten Schicht `fc` an. Was dort anliegt, ist das Embedding.
**Lücke:** die letzte Schicht anwenden – `logits_hand = modell.fc(emb_hand)`.
**Teil C – derselbe Durchlauf mit Hook**, wie im Dienst.

**Erwartung:** Embedding mit **512** Werten; von Hand und mit Hook identisch; die Klasse ist „Kirche“.

In [ ]:
%%nur_mit_dienst
# ---------- Teil A: Mini-Netz aus drei Schritten ---------------------------------------------------------------
def verdoppeln(x): return x * 2
def plus_eins(x):  return x + 1
def quadrat(x):    return x ** 2
mini_netz = [("verdoppeln", verdoppeln), ("plus_eins", plus_eins), ("quadrat", quadrat)]   # Liste der Schichten, in Reihenfolge

notiz = {}                                                  # hier schreibt der "Hook" hinein
def durchlauf(x, hook_nach=None):
    for name, schicht in mini_netz:                         # eine Schicht nach der anderen
        x = schicht(x)
        if name == hook_nach:                               # Hook: Zwischenergebnis merken ...
            notiz["v"] = x                                  # ... das Ergebnis selbst bleibt unveraendert
    return x

print("Mini-Netz, Eingabe 3:  ohne Hook ->", durchlauf(3), "| mit Hook nach 'plus_eins' ->", durchlauf(3, "plus_eins"),
      "| gemerkt:", notiz["v"])

# ---------- Teil B: das Traegermodell Schicht fuer Schicht -----------------------------------------------------
bild = Image.open(BILD_KIRCHE).convert("RGB")
x = tf(bild).unsqueeze(0)                                   # Vorverarbeitung wie im Dienst, plus Batch-Achse: (1, 3, 160, 160)
print(f"\n{'Schicht':10s} Form der Ausgabe")
with torch.no_grad():
    for name, schicht in modell.named_children():           # die zehn Bausteine von ResNet-18, in Reihenfolge
        if name == "fc":                                    # vor der letzten Schicht anhalten
            break
        x = schicht(x)
        print(f"{name:10s} {tuple(x.shape)}")
    emb_hand = x.flatten(1)                                 # (1, 512, 1, 1) -> (1, 512): das Embedding
    # TODO 1.1c: die letzte Schicht anwenden: logits_hand = modell.fc(emb_hand)
    raise NotImplementedError("TODO 1.1c")
p_hand = torch.softmax(logits_hand, dim=1)[0]
print(f"{'fc':10s} {tuple(logits_hand.shape)}   -> Klasse von Hand: {KLASSEN[int(p_hand.argmax())]} (p = {float(p_hand.max()):.4f})")
print("Embedding: ", emb_hand.shape[1], "Werte, die ersten fuenf:", np.round(emb_hand[0, :5].numpy(), 4))

# ---------- Teil C: derselbe Durchlauf mit Hook (wie im Dienst) --------------------------------------------------
emb = {}
def merke(schicht, eingabe, ausgabe):
    emb["v"] = ausgabe.flatten(1)
griff = modell.avgpool.register_forward_hook(merke)         # anmelden; "griff" brauchen wir zum Abmelden
with torch.no_grad():
    logits_hook = modell(tf(bild).unsqueeze(0))             # ganz normaler Aufruf - der Hook laeuft nebenbei mit
griff.remove()                                              # Hook wieder abmelden (im Dienst bleibt er angemeldet)
print("mit Hook:  ", emb["v"].shape[1], "Werte, die ersten fuenf:", np.round(emb["v"][0, :5].numpy(), 4))

check("1.1 PRINZIP: Mini-Netz - Hook merkt 7, Ergebnis bleibt 49", lambda: notiz["v"] == 7 and durchlauf(3) == 49)
check("1.1 PRINZIP: Embedding hat 512 Werte", lambda: emb_hand.shape == (1, 512))
check("1.1 PRINZIP: von Hand und mit Hook dasselbe Embedding und dieselben Logits",
      lambda: torch.allclose(emb_hand, emb["v"]) and torch.allclose(logits_hand, logits_hook, atol=1e-5))
check("1.1 PRINZIP: Klasse von Hand ist Kirche", lambda: KLASSEN[int(p_hand.argmax())] == "Kirche")

### 1.2 · Starten, ein Bild schicken, die Log-Zeile lesen

`CODE` (2 Zeilen) · Die Zelle startet den Dienst im Hintergrund – im Terminal hieße das `uvicorn lab08.app:app` –,
schickt das Kirchenbild an `/predict` und liest danach die **letzte Zeile** des Protokolls.

**Lücke:** Die Datei `LOGDATEI` als Text lesen, in Zeilen zerlegen, die letzte nehmen und aus dem JSON-Text ein
Dictionary machen:

- `letzte = LOGDATEI.read_text(encoding="utf-8").splitlines()[-1]`
- `eintrag = json.loads(letzte)`

**Erwartung:** Status `200`, Antwort mit genau den drei Feldern aus Lab 7; die Log-Zeile hat vier Schlüssel, fünf
Wahrscheinlichkeiten mit Summe 1 und 512 Embedding-Werte.

In [ ]:
%%nur_mit_dienst
PORT = freier_port(8000, 8001, 8002, 8003)
URL = starte("dienst", f"uvicorn lab08.app:app --port {PORT}", PORT, pfad="/health", umgebung={"MODEL_URI": MODEL_URI})

anfrage = {"image_b64": base64.b64encode(BILD_KIRCHE.read_bytes()).decode()}     # Bilddatei -> Base64-Text
status, antwort = frage(URL, "/predict", anfrage)

# TODO 1.2: letzte = letzte Zeile von LOGDATEI (read_text, splitlines, [-1]) und eintrag = json.loads(letzte)
raise NotImplementedError("TODO 1.2")

print("\nLog-Zeile, Schluessel:   ", list(eintrag))
print("t (Zeitpunkt):           ", eintrag["t"], "=", time.strftime("%d.%m.%Y %H:%M:%S", time.localtime(eintrag["t"])))
print("klasse:                  ", eintrag["klasse"])
print("p (5 Werte):             ", eintrag["p"], "| Summe:", round(sum(eintrag["p"]), 4))
print("emb:                     ", len(eintrag["emb"]), "Werte, die ersten fuenf:", eintrag["emb"][:5])
kb = len(letzte.encode("utf-8")) / 1000
print(f"Groesse der Zeile:        {kb:.1f} kB  ->  bei 1 Mio. Anfragen am Tag: {kb:.1f} GB Protokoll pro Tag")

check("1.2 Dienst antwortet: Status 200, Klasse Kirche", lambda: status == 200 and antwort["klasse"] == "Kirche")
check("1.2 Antwort hat genau die Felder aus Lab 7", lambda: set(antwort) == {"klasse", "p", "modell"})
check("1.2 Log-Zeile hat die vier Schluessel", lambda: set(eintrag) == {"t", "klasse", "p", "emb"})
check("1.2 Fuenf Wahrscheinlichkeiten mit Summe 1, 512 Embedding-Werte",
      lambda: len(eintrag["p"]) == 5 and nah(sum(eintrag["p"]), 1.0, 1e-3) and len(eintrag["emb"]) == 512)
check("1.2 Groesste Wahrscheinlichkeit im Log = p in der Antwort", lambda: nah(max(eintrag["p"]), antwort["p"], 2e-4))

*Ihre Antworten (`DENKEN`):*

1. Was steht im Log, das die Antwort an den Client **nicht** verrät? Wozu könnte man es brauchen?
2. Im Log steht **nicht**, ob die Antwort richtig war. Warum nicht – und wer könnte das später nachtragen?
3. Eine Zeile ist einige Kilobyte groß. Was würden Sie in einem Dienst mit einer Million Anfragen am Tag protokollieren –
   alles, eine Stichprobe, nur die Konfidenz? Was verlieren Sie jeweils?

## Aufgabe 2 · Zwei Bildströme durch den Dienst

**In der Vorlesung: Live 1, Schritte 2, 3 und 5 · ≈ 20 Min.** · Folien „Die Störungen unseres Labs“ und Anhang B
„Code: Drift-Strom senden“

Wir spielen Produktion: Die **501 Validierungsbilder** gehen zweimal durch den Dienst.

- **Referenzstrom:** unverändert. Auf diesen Bildern kennen wir die Qualität des Modells (Genauigkeit 0,984).
- **Driftstrom:** dieselben Bilder, aber so, als wäre in der Halle das Licht heller geworden, die Kamera leicht
  unscharf gestellt und die Übertragung auf starke Kompression umgestellt worden. Die Motive bleiben dieselben – es ändert
  sich nur, **wie die Bilder aussehen**. Das ist ein *Covariate Shift*: $P(X)$ ändert sich, $P(Y \mid X)$ bleibt.

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 2.1 | Die drei Störungen von Hand | `PRINZIP` + `CODE` (1 Zeile) | 7 Min. |
| 2.2 | `lab08/senden.py` vervollständigen und beide Ströme senden | `CODE` (3 Zeilen) | 7 Min. |
| 2.3 | Zwei Bilder, je zwei Antworten | `AUSFÜHREN` | 2 Min. |
| 2.4 | Zwei Fenster aus dem Log laden | `AUSFÜHREN` | 4 Min. |

### 2.1 `PRINZIP` · Die drei Störungen von Hand

`PRINZIP` + `CODE` (1 Zeile) · Ein Bild ist für den Rechner eine Tabelle von Zahlen: 160 × 160 Pixel, je Pixel drei
Werte (Rot, Grün, Blau) zwischen 0 und 255. Alle drei Störungen sind Rechenvorschriften auf diesen Zahlen:

- **Helligkeit × 1,6:** jeden Wert mit 1,6 multiplizieren; was über 255 hinausgeht, wird auf 255 gekappt („ausgebrannt“).
  **Lücke:** `heller = np.clip(x * 1.6, 0, 255).astype(np.uint8)`
- **Unschärfe (Gaussian Blur, 5 Pixel breit, σ = 1,5):** Jeder Pixel wird durch einen gewichteten Mittelwert seiner
  Nachbarn ersetzt. Die Gewichte folgen der Glockenkurve $g(d) = e^{-d^2 / (2\sigma^2)}$ für die Abstände
  $d = -2, \dots, 2$, geteilt durch ihre Summe.
- **JPEG mit Qualität 20:** Das Bild wird in 8 × 8-Blöcke zerlegt und stark vereinfacht gespeichert. Wir rechnen das
  nicht nach, sondern messen die Wirkung: Dateigröße und Abweichung vom Original.

Am Ende steht die Funktion `stoere(bild)`, die alle drei Störungen mit torchvision hintereinander ausführt – dieselbe
Kette benutzt gleich das Sende-Skript.

**Erwartung:** Helligkeit von Hand = torchvision (Abweichung höchstens 1 Stufe durch Rundung); die fünf Gewichte
addieren sich zu 1; die JPEG-Datei mit Qualität 20 ist deutlich kleiner als mit Qualität 95.

In [ ]:
abschnitt("Aufgabe 2")
bild = Image.open(BILD_KIRCHE).convert("RGB")
x = np.array(bild)                                          # Bild als Zahlentabelle: (160, 160, 3), Werte 0 ... 255
print("Form:", x.shape, "| Datentyp:", x.dtype, "| Pixel oben links (R, G, B):", x[0, 0])

# ---------- Helligkeit von Hand ----------------------------------------------------------------------------------
# TODO 2.1: heller = np.clip(x * 1.6, 0, 255).astype(np.uint8)
raise NotImplementedError("TODO 2.1")
print(f"Helligkeit x 1,6:  Pixel oben links {x[0, 0]} -> {heller[0, 0]} | "
      f"Anteil 'ausgebrannter' Werte (= 255): vorher {(x == 255).mean():.1%}, nachher {(heller == 255).mean():.1%}")
heller_tv = F.adjust_brightness(F.pil_to_tensor(bild), 1.6).permute(1, 2, 0).numpy()     # dasselbe mit torchvision
print("   groesste Abweichung von Hand gegen torchvision:", int(np.abs(heller.astype(int) - heller_tv.astype(int)).max()), "Stufe(n)")

# ---------- Unschaerfe: die fuenf Gewichte von Hand ----------------------------------------------------------------
sigma = 1.5
roh = [math.exp(-(d * d) / (2 * sigma * sigma)) for d in (-2, -1, 0, 1, 2)]      # Glockenkurve an fuenf Stellen
gewichte = [g / sum(roh) for g in roh]                                           # durch die Summe teilen -> Summe 1
print("\nGewichte der Unschaerfe (Abstand -2 ... +2):", [round(g, 3) for g in gewichte], "| Summe:", round(sum(gewichte), 6))
zeile = x[80, 60:70, 0].astype(float)                       # zehn Rot-Werte aus der Bildmitte
glatt = []
for i in range(2, len(zeile) - 2):                          # fuer jeden Pixel mit zwei Nachbarn links und rechts
    glatt.append(sum(gewichte[j] * zeile[i - 2 + j] for j in range(5)))          # gewichteter Mittelwert der fuenf Nachbarn
print("   zehn Rot-Werte einer Bildzeile:", zeile.astype(int))
print("   geglaettet (ohne die Raender): ", np.round(glatt).astype(int), "<- Spruenge werden kleiner")

# ---------- JPEG: Wirkung messen -------------------------------------------------------------------------------------
def jpeg_bytes(b, qualitaet):
    puffer = io.BytesIO(); b.save(puffer, format="JPEG", quality=qualitaet); return puffer.getvalue()
gross, klein = jpeg_bytes(bild, 95), jpeg_bytes(bild, 20)
print(f"\nJPEG Qualitaet 95: {len(gross) / 1000:.1f} kB | Qualitaet 20: {len(klein) / 1000:.1f} kB "
      f"({len(klein) / len(gross):.0%} der Groesse)")

# ---------- Die ganze Kette mit torchvision (dieselbe wie gleich im Sende-Skript) ------------------------------------
def stoere(bild, helligkeit=1.6, sigma=1.5, qualitaet=20):
    """PIL-Bild stoeren: heller, unscharf, JPEG-komprimiert. helligkeit=1, sigma=0, qualitaet=100 schalten je eine Stoerung ab."""
    t = F.pil_to_tensor(bild)                               # Bild -> Tensor mit ganzen Zahlen 0 ... 255 (noetig fuer JPEG)
    if helligkeit != 1: t = F.adjust_brightness(t, helligkeit)
    if sigma > 0:       t = F.gaussian_blur(t, kernel_size=5, sigma=sigma)       # Kerngroesse muss ungerade sein
    if qualitaet < 100: t = F.jpeg(t, quality=qualitaet)                         # 1 = staerkste, 100 = schwaechste Kompression
    return F.to_pil_image(t)                                # Tensor -> Bild

varianten = {"Original": bild, "Helligkeit × 1,6": stoere(bild, sigma=0, qualitaet=100),
             "Unschärfe σ = 1,5": stoere(bild, helligkeit=1, qualitaet=100), "JPEG Qualität 20": stoere(bild, helligkeit=1, sigma=0),
             "alle drei (Driftstrom)": stoere(bild)}
fig, achsen = plt.subplots(1, 5, figsize=(13, 2.9))
for ax, (titel, b) in zip(achsen, varianten.items()):
    ax.imshow(b); ax.set_title(titel, fontsize=10, color=ORANGE if "alle" in titel else NAVY); ax.axis("off")
plt.tight_layout(); plt.savefig(RESULTS / "stoerungen.png", dpi=150); plt.show()

check("2.1 Helligkeit von Hand = torchvision (hoechstens 1 Stufe Abweichung)", lambda: np.abs(heller.astype(int) - heller_tv.astype(int)).max() <= 1)
check("2.1 Fuenf Gewichte, Summe 1, das mittlere ist das groesste", lambda: nah(sum(gewichte), 1, 1e-9) and max(gewichte) == gewichte[2])
check("2.1 JPEG mit Qualitaet 20 ist kleiner als die Haelfte von Qualitaet 95", lambda: len(klein) < 0.5 * len(gross))
check("2.1 stoere() liefert wieder ein Bild mit 160 x 160 Pixeln", lambda: stoere(bild).size == (160, 160))

*Ihre Beobachtung (`DENKEN`):* Welche der drei Störungen sehen Sie mit bloßem Auge sofort, welche kaum? Würden Sie als
Prüfer am Band das gestörte Bild noch richtig einordnen?

### 2.2 · `lab08/senden.py` vervollständigen und beide Ströme senden

`CODE` (3 Zeilen) · Das Skript liest `data/imagenette5/val.txt` (je Zeile: Pfad und Klassenindex), öffnet jedes Bild,
stört es – nur beim Driftstrom – und schickt es als Base64-Text an `/predict`. Am Ende schreibt es eine kleine Datei
`results/fenster_<strom>.json` mit **Start- und Endzeit** des Stroms. Damit finden wir später im Protokoll genau die
Zeilen dieses Stroms: Ein **Fenster** ist ein Zeitbereich.

**Lücke:** Die Kette `DRIFT` beginnt mit `T.PILToTensor()` und endet mit `T.ToPILImage()`. Dazwischen fehlen die drei
Störungen – je eine Zeile `schritte.append(...)`:

- `T.Lambda(lambda x: F.adjust_brightness(x, 1.6))` – heller
- `T.GaussianBlur(kernel_size=5, sigma=1.5)` – unscharf
- `T.JPEG(quality=20)` – komprimiert

Die zweite Zelle sendet beide Ströme. **Erwartung:** je **501 Bilder**, je Strom wenige Sekunden bis etwa eine halbe
Minute (je nach Rechner).

In der Fenster-Datei stehen auch die **richtigen Klassen** der gesendeten Bilder. Das ist die Abkürzung dieses Labs: Im
Betrieb kämen sie erst später – von einem Prüfer. Wir legen sie beiseite und benutzen sie erst in Aufgabe 4.2.

In [ ]:
%%nur_mit_dienst
schreibe("lab08/senden.py", r"""
# lab08/senden.py - schickt die Validierungsbilder an den Dienst: unveraendert (referenz) oder gestoert (drift)
# Aufruf (aus dem Hauptordner des Repos, der Dienst muss laufen):  python lab08/senden.py --strom referenz
import argparse, base64, io, json, os, time

import httpx
from PIL import Image
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as F

# ---- die Stoerungen des Driftstroms: eine Kette von Bildoperationen ---------------------------------------------
schritte = [T.PILToTensor()]                       # Bild -> Tensor mit ganzen Zahlen 0 ... 255 (noetig fuer JPEG)
# TODO 2.2: drei Zeilen schritte.append(...): Helligkeit 1.6, GaussianBlur(kernel_size=5, sigma=1.5), JPEG(quality=20)
raise NotImplementedError("TODO 2.2")
schritte.append(T.ToPILImage())                    # Tensor -> Bild
DRIFT = T.Compose(schritte)                        # alle Schritte hintereinander


def b64(bild):
    puffer = io.BytesIO()
    bild.save(puffer, format="PNG")                # PNG ist verlustfrei - es kommt keine zweite Kompression dazu
    return base64.b64encode(puffer.getvalue()).decode()


arg = argparse.ArgumentParser()
arg.add_argument("--strom", choices=["referenz", "drift"], required=True)
arg.add_argument("--url", default="http://127.0.0.1:8000")
a = arg.parse_args()

zeilen = open("data/imagenette5/val.txt").read().strip().splitlines()     # je Zeile: Pfad <TAB> Klassenindex
labels = []
with httpx.Client(base_url=a.url, timeout=60, trust_env=False) as client:  # trust_env=False: nie ueber einen Proxy
    t0 = time.time()                                                       # Beginn des Fensters
    for nr, zeile in enumerate(zeilen, 1):
        pfad, label = zeile.split("\t")
        bild = Image.open("data/imagenette5/" + pfad).convert("RGB")
        if a.strom == "drift":
            bild = DRIFT(bild)                                             # nur der Driftstrom wird gestoert
        client.post("/predict", json={"image_b64": b64(bild)}).raise_for_status()   # Fehlercode -> Abbruch mit Meldung
        labels.append(int(label))
        if nr % 100 == 0:
            print(f"{nr} Bilder gesendet")
    t1 = time.time()                                                       # Ende des Fensters

# Fenster festhalten: Zeitbereich des Stroms und - nur fuer Aufgabe 4.2 - die richtigen Klassen der Bilder
os.makedirs("results", exist_ok=True)
with open(f"results/fenster_{a.strom}.json", "w") as f:
    json.dump({"strom": a.strom, "t0": t0, "t1": t1, "n": len(labels), "labels": labels}, f)
print(f"Strom {a.strom}: {len(labels)} Bilder in {t1 - t0:.1f} s gesendet -> results/fenster_{a.strom}.json")
""")
check("2.2 senden.py geschrieben, drei Stoerungen in der Kette",
      lambda: all(s in (ARBEIT / "lab08/senden.py").read_text(encoding="utf-8") for s in ("adjust_brightness", "GaussianBlur", "JPEG(")))

In [ ]:
%%nur_mit_dienst
STROM = {}
for strom in ("referenz", "drift"):
    STROM[strom] = run(f"python lab08/senden.py --strom {strom} --url {URL}")
    print()
fenster = {s: json.loads((ARBEIT / "results" / f"fenster_{s}.json").read_text()) for s in ("referenz", "drift")}
for s, f in fenster.items():
    print(f"Fenster {s:9s}: {time.strftime('%H:%M:%S', time.localtime(f['t0']))} bis {time.strftime('%H:%M:%S', time.localtime(f['t1']))}"
          f" = {f['t1'] - f['t0']:.1f} s, {f['n']} Bilder, {1000 * (f['t1'] - f['t0']) / f['n']:.0f} ms je Anfrage")
check(f"2.2 Beide Stroeme vollstaendig gesendet (je {len(VAL)} Bilder)", lambda: all(f["n"] == len(VAL) for f in fenster.values()))
check("2.2 Der Driftstrom begann nach dem Ende des Referenzstroms", lambda: fenster["drift"]["t0"] > fenster["referenz"]["t1"])

### 2.3 · Zwei Bilder, je zwei Antworten

`AUSFÜHREN` · Zwei Kirchenbilder gehen je zweimal an den Dienst: unverändert und gestört. Danach zählt die Zelle die
Zeilen des Protokolls.

**Vorher überlegen:** Ändert sich die Klasse – oder nur `p`? Und wie viele Zeilen hat das Log nach beiden Strömen
mindestens?

In [ ]:
%%nur_mit_dienst
BILD_KIRCHE_2 = BILDER / "val" / "n03028079" / "n03028079_122592.JPEG"     # ein zweites Kirchenbild
ZWEI = {}
fig, achsen = plt.subplots(1, 4, figsize=(10.5, 2.9))
for nr, pfad in enumerate((BILD_KIRCHE, BILD_KIRCHE_2)):
    bild = Image.open(pfad).convert("RGB")
    for j, (name, b) in enumerate((("unveraendert", bild), ("gestoert", stoere(bild)))):
        status, antw = frage(URL, "/predict", {"image_b64": b64_png(b)}, zeigen=False)
        ZWEI[f"Kirche {nr + 1} {name}"] = {"status": status, **antw}
        print(f"Kirche {nr + 1} {name:13s} -> {status}: Klasse {antw['klasse']}, p = {antw['p']}")
        ax = achsen[2 * nr + j]; ax.imshow(b); ax.axis("off")
        ax.set_title(f"{antw['klasse']} · p = {antw['p']:.3f}", fontsize=10, color=NAVY if antw["klasse"] == "Kirche" else ORANGE)
plt.tight_layout(); plt.savefig(RESULTS / "zwei_antworten.png", dpi=150); plt.show()
n_zeilen = len(LOGDATEI.read_text(encoding="utf-8").splitlines())
print(f"\nDas Protokoll hat jetzt {n_zeilen} Zeilen ({LOGDATEI.stat().st_size / 1e6:.1f} MB): 2 x {len(VAL)} aus den Stroemen plus Ihre einzelnen Anfragen.")
check("2.3 Alle vier Anfragen wurden mit Status 200 beantwortet", lambda: all(a["status"] == 200 for a in ZWEI.values()))
check("2.3 Unveraendert erkennt der Dienst beide Kirchen", lambda: all(a["klasse"] == "Kirche" for k, a in ZWEI.items() if "unveraendert" in k))
check(f"2.3 Das Protokoll hat mindestens {2 * len(VAL)} Zeilen", lambda: n_zeilen >= 2 * len(VAL))

*Ihre Beobachtung (`DENKEN`):* Der Dienst hat alle vier Male mit `200` geantwortet. Woran hätte ein Betreiber, der nur auf
Statuscodes und Antwortzeiten schaut, gemerkt, dass sich etwas geändert hat?

### 2.4 · Zwei Fenster aus dem Log laden

`AUSFÜHREN` · Ab hier arbeiten wir **nur noch mit dem Protokoll** – so, wie es ein Monitoring-Job täte, der einmal am Tag
läuft und den Dienst nie zu sehen bekommt. Die Funktion `lade(strom)` geht das Log Zeile für Zeile durch und behält die
Einträge, deren Zeitpunkt `t` im Fenster des Stroms liegt.

Ergebnis je Fenster:

- `P` – Matrix **501 × 5**: die fünf Wahrscheinlichkeiten je Anfrage
- `E` – Matrix **501 × 512**: das Embedding je Anfrage
- `konf` – 501 Werte: die **Konfidenz** je Anfrage, also die größte der fünf Wahrscheinlichkeiten (`P.max(axis=1)`)

**Kein Dienst?** Dann lädt die Zelle dieselben Matrizen aus `lab08/vorberechnet/` (Referenzlauf). Ab hier ist das Lab
für alle gleich.

In [ ]:
def lade(strom):
    """Fenster eines Stroms aus dem Protokoll laden: (P, E, labels). Liest results/fenster_<strom>.json und das Log."""
    f = json.loads((ARBEIT / "results" / f"fenster_{strom}.json").read_text())   # Zeitbereich des Stroms
    P, E = [], []
    with open(LOGDATEI, encoding="utf-8") as log:
        for zeile in log:                                   # das Protokoll Zeile fuer Zeile
            e = json.loads(zeile)                           # JSON-Text -> Dictionary
            if f["t0"] <= e["t"] <= f["t1"]:                # liegt der Zeitpunkt im Fenster?
                P.append(e["p"]); E.append(e["emb"])
    return np.array(P), np.array(E), np.array(f["labels"])

def lade_vorberechnet(strom):
    """Dieselben Matrizen aus dem Referenzlauf (lab08/vorberechnet/log_<strom>.npz)."""
    z = np.load(VORBERECHNET / f"log_{strom}.npz")
    return z["p"].astype(float), z["emb"].astype(float), z["labels"]

EIGENE_LOGS = DIENST_MOEGLICH and all((ARBEIT / "results" / f"fenster_{s}.json").exists() for s in ("referenz", "drift")) and LOGDATEI.exists()
quelle = lade if EIGENE_LOGS else lade_vorberechnet
P_ref, E_ref, y_ref = quelle("referenz")                    # Referenzfenster
P_akt, E_akt, y_akt = quelle("drift")                       # aktuelles Fenster (hier: der Driftstrom)
konf_ref, konf_akt = P_ref.max(axis=1), P_akt.max(axis=1)   # Konfidenz = groesste Wahrscheinlichkeit je Zeile

print("Quelle:", "Ihr Protokoll " + str(LOGDATEI.relative_to(ARBEIT)) if EIGENE_LOGS else "vorberechnete Logs aus lab08/vorberechnet/")
print(f"Referenz: P {P_ref.shape}, E {E_ref.shape} | aktuell: P {P_akt.shape}, E {E_akt.shape}")
print(f"mittlere Konfidenz:   Referenz {konf_ref.mean():.3f} | aktuell {konf_akt.mean():.3f}")
print(f"Konfidenz unter 0,6:  Referenz {(konf_ref < 0.6).mean():.1%} | aktuell {(konf_akt < 0.6).mean():.1%}")
print("Die richtigen Klassen (y_ref, y_akt) legen wir beiseite - im Betrieb haetten wir sie noch nicht.")

N = len(P_ref)                                              # Anzahl der Anfragen je Fenster (500 oder 501)
check(f"2.4 Beide Fenster vollstaendig: je {len(y_ref)} Zeilen, 5 Wahrscheinlichkeiten, 512 Embedding-Werte",
      lambda: N in (500, 501) and P_ref.shape == P_akt.shape == (len(y_ref), 5) and E_ref.shape == E_akt.shape == (len(y_ref), 512),
      "Stroeme in 2.2 erneut senden (jeder Lauf schreibt sein Fenster neu)")
check("2.4 Die mittlere Konfidenz ist im Driftstrom niedriger", lambda: konf_akt.mean() < konf_ref.mean())

*Ihre Beobachtung (`DENKEN`):* Die mittlere Konfidenz ist gesunken. Reicht das als Alarm? Was fehlt Ihnen, um zu
entscheiden, ob der Unterschied „groß“ ist?

## Aufgabe 3 · Drift-Detektor selbst bauen

**In der Vorlesung: Live 2, Schritte 1–3 · ≈ 35 Min.** · Folien „PSI“, „Beispiel: PSI von Hand“, „KS-Test“,
„Beispiel: KS-Statistik ablesen“, „Bilder haben 512 Dimensionen“ und Anhang B

Zwei Fenster, keine Labels. Die Frage an beide Stichproben lautet: **Stammen sie aus derselben Verteilung?** Sie bauen
zwei Antworten – eine Kennzahl und einen Test:

| | PSI (Population Stability Index) | KS-Test (Kolmogorov-Smirnov) |
|---|---|---|
| vergleicht | Anteile in festen Klassen (Bins) | die ganzen Verteilungsfunktionen |
| Ergebnis | eine Kennzahl, Faustschwellen 0,1 und 0,25 | Statistik $D$ und ein p-Wert |
| hier angewandt auf | die Konfidenz (1 Spalte) | Konfidenz und 512 Embedding-Dimensionen |

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 3.1 | PSI von Hand und als Funktion | `PRINZIP` + `CODE` (1 Zeile) | 10 Min. |
| 3.2 | KS-Statistik von Hand | `PRINZIP` + `CODE` (2 Zeilen) | 10 Min. |
| 3.3 | 512 Tests auf einmal – und warum man korrigieren muss | `PRINZIP` + `CODE` (1 Zeile) | 8 Min. |
| 3.4 | Gegenprobe: Referenz gegen Referenz | `CODE` (1 Zeile) | 7 Min. |

### 3.1 · PSI von Hand und als Funktion

`PRINZIP` + `CODE` (1 Zeile) · Der PSI teilt die Werte in Klassen ein und vergleicht je Klasse den Anteil in der Referenz
($p_i$) mit dem Anteil im aktuellen Fenster ($q_i$):

$$\mathrm{PSI} = \sum_i \,(q_i - p_i) \cdot \ln\frac{q_i}{p_i}$$

**So liest man die Formel:** Für jede Klasse: *Differenz der Anteile* mal *Logarithmus ihres Verhältnisses*. Beide Faktoren
haben immer dasselbe Vorzeichen – jeder Summand ist also ≥ 0, und der PSI ist genau dann 0, wenn alle Anteile gleich sind.
Der Logarithmus gewichtet: Eine Verdopplung von 5 % auf 10 % zählt mehr als ein Rückgang von 80 % auf 75 %.

Unsere vier Klassen der Konfidenz (wie auf der Folie): **unter 0,6 · 0,6 bis unter 0,8 · 0,8 bis unter 0,95 · ab 0,95**.

- **Teil A** rechnet das Folienbeispiel in einer Schleife nach (Referenz 5 / 5 / 10 / 80 %, Produktion 10 / 15 / 25 / 50 %).
- **Teil B** zählt die Anteile Ihrer beiden Fenster von Hand aus.
- **Teil C** ist die Funktion für den Alltag. **Lücke** in `psi_aus_anteilen`: die Formel in einer Zeile –
  `return float(np.sum((q - p) * np.log(q / p)))`

**Faustregel** (aus dem Kreditgeschäft, keine statistische Herleitung): unter 0,1 kaum Veränderung · 0,1 bis 0,25 prüfen ·
über 0,25 handeln.

**Erwartung:** Folienbeispiel **0,423**; Ihre Fenster: PSI ≈ **0,40** – über 0,25.

In [ ]:
abschnitt("Aufgabe 3")
# ---------- Teil A: das Folienbeispiel in einer Schleife ---------------------------------------------------------
klassen_namen = ["unter 0,6", "0,6 bis 0,8", "0,8 bis 0,95", "ab 0,95"]
p_folie = [0.05, 0.05, 0.10, 0.80]                          # Anteile Referenz
q_folie = [0.10, 0.15, 0.25, 0.50]                          # Anteile Produktion
print(f"{'Klasse':14s} {'p (Ref.)':>9s} {'q (akt.)':>9s} {'q - p':>8s} {'ln(q/p)':>9s} {'Summand':>9s}")
psi_hand = 0.0
for name, p, q in zip(klassen_namen, p_folie, q_folie):     # eine Klasse nach der anderen
    summand = (q - p) * math.log(q / p)                     # Differenz mal Logarithmus des Verhaeltnisses
    psi_hand += summand
    print(f"{name:14s} {p:9.2f} {q:9.2f} {q - p:+8.2f} {math.log(q / p):+9.3f} {summand:9.3f}")
print(f"{'PSI':14s} {'':37s} {psi_hand:9.3f}   (Folie: 0,423)")

# ---------- Teil B: Anteile der eigenen Fenster von Hand auszaehlen ------------------------------------------------
def anteile_von_hand(werte):
    zaehler = [0, 0, 0, 0]
    for w in werte:                                         # jeden Wert in seine Klasse einsortieren
        if   w < 0.6:  zaehler[0] += 1
        elif w < 0.8:  zaehler[1] += 1
        elif w < 0.95: zaehler[2] += 1
        else:          zaehler[3] += 1
    return [z / len(werte) for z in zaehler]                # Anzahl -> Anteil

a_ref, a_akt = anteile_von_hand(konf_ref), anteile_von_hand(konf_akt)
print("\nAnteile Referenz:", [f"{a:.1%}" for a in a_ref])
print("Anteile aktuell: ", [f"{a:.1%}" for a in a_akt])

# ---------- Teil C: die Funktionen fuer den Alltag -----------------------------------------------------------------
GRENZEN = [0.6, 0.8, 0.95]                                  # feste Klassengrenzen wie im Beispiel

def anteile(x, grenzen=GRENZEN):
    nummer = np.digitize(x, grenzen)                        # Klassennummer je Wert: 0 (unter 0,6) ... 3 (ab 0,95)
    return np.bincount(nummer, minlength=len(grenzen) + 1) / len(x)    # je Klasse zaehlen, durch die Anzahl teilen

def psi_aus_anteilen(p, q, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, None)      # leere Klassen: kleiner Mindestwert statt 0 (sonst ln(0))
    q = np.clip(np.asarray(q, dtype=float), eps, None)
    # TODO 3.1: die Formel in einer Zeile: return float(np.sum((q - p) * np.log(q / p)))
    raise NotImplementedError("TODO 3.1")

def psi(ref, akt, grenzen=GRENZEN):
    return psi_aus_anteilen(anteile(ref, grenzen), anteile(akt, grenzen))

PSI_KONF = psi(konf_ref, konf_akt)
print(f"\nPSI der Konfidenz, Referenz gegen aktuell: {PSI_KONF:.3f}  ->",
      "ueber 0,25: handeln" if PSI_KONF > 0.25 else "0,1 bis 0,25: pruefen" if PSI_KONF > 0.1 else "unter 0,1: kaum Veraenderung")

fig, ax = plt.subplots(figsize=(6.4, 3.1))
pos = np.arange(4)
ax.bar(pos - 0.2, anteile(konf_ref), 0.4, color=NAVY, label="Referenz")
ax.bar(pos + 0.2, anteile(konf_akt), 0.4, color=ORANGE, label="aktuell (Drift)")
ax.set_xticks(pos); ax.set_xticklabels(klassen_namen); ax.set_ylabel("Anteil"); ax.set_xlabel("Konfidenz")
ax.set_title(f"Konfidenz in vier Klassen · PSI = {PSI_KONF:.3f}", color=NAVY, fontsize=11); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(RESULTS / "psi_konfidenz.png", dpi=150); plt.show()

check("3.1 Folienbeispiel von Hand: PSI = 0,423", lambda: nah(psi_hand, 0.423, 0.0006))
check("3.1 psi_aus_anteilen() liefert fuer das Folienbeispiel dasselbe", lambda: nah(psi_aus_anteilen(p_folie, q_folie), psi_hand, 1e-9))
check("3.1 Anteile von Hand = Anteile mit numpy", lambda: np.allclose(a_ref, anteile(konf_ref)) and np.allclose(a_akt, anteile(konf_akt)))
check("3.1 Gleiche Verteilung -> PSI = 0", lambda: psi(konf_ref, konf_ref) == 0)
check("3.1 PSI der Konfidenz liegt ueber 0,25 (Referenzlauf: 0,400)", lambda: 0.25 < PSI_KONF < 0.60)

*Ihre Antworten (`DENKEN`):*

1. Welche der vier Klassen trägt am meisten zum PSI bei – die, in der sich die meisten Anfragen verschoben haben?
2. Der PSI hängt von den Klassengrenzen ab. Was passiert, wenn Sie nur **zwei** Klassen bilden? Probieren Sie in einer
   neuen Zelle `psi(konf_ref, konf_akt, grenzen=[0.95])` und `psi(konf_ref, konf_akt, grenzen=[0.6])`.
3. Warum ist der PSI **kein** statistischer Test?

### 3.2 · KS-Statistik von Hand

`PRINZIP` + `CODE` (2 Zeilen) · Der KS-Test braucht keine Klassen. Er vergleicht die **empirischen
Verteilungsfunktionen**: $F(x)$ = Anteil der Werte, die höchstens $x$ groß sind – eine Treppe, die von 0 auf 1 steigt.
Die Statistik ist der **größte senkrechte Abstand** der beiden Treppen:

$$D = \max_x \,\bigl|\,F_\text{ref}(x) - F_\text{akt}(x)\,\bigr|$$

**So liest man die Formel:** Gehe alle Stellen $x$ durch, miss dort den Abstand der beiden Treppen, nimm den größten.
$D$ liegt zwischen 0 (Treppen deckungsgleich) und 1 (alle Werte der einen Stichprobe liegen unter allen der anderen).
Aus $D$ und den Stichprobengrößen folgt der **p-Wert**: die Wahrscheinlichkeit, einen mindestens so großen Abstand zu
sehen, **wenn beide Stichproben aus derselben Verteilung stammen**. Kleiner p-Wert → der Unterschied ist kaum Zufall.

- **Teil A** rechnet das Folienbeispiel nach: 5 Konfidenzen Referenz, 6 Produktion.
  **Lücke** in `ks_von_hand`: Wenn `abstand` größer ist als das bisherige `D`, merken Sie sich beides –
  `D = abstand` und `wo = x`.
- **Teil B** wendet dieselbe Funktion auf Ihre 501 + 501 Konfidenzen an und vergleicht mit `scipy.stats.ks_2samp`.

**Erwartung:** Folienbeispiel $D$ = **0,467** bei $x$ = 0,78 mit p = 0,47 (zu wenige Punkte für eine Aussage).
Ihre Fenster: $D$ ≈ **0,40** mit einem p-Wert, der praktisch 0 ist. Der größte Abstand liegt ganz rechts, kurz vor 1 –
das Modell ist auf der Referenz bei sehr vielen Bildern „praktisch sicher“.

In [ ]:
def ks_von_hand(a, b, zeigen=False):
    """Groesster Abstand der beiden Treppen. Rueckgabe: (D, Stelle x, an der er auftritt)."""
    D, wo = 0.0, None
    for x in sorted(list(a) + list(b)):                     # alle Werte beider Stichproben, aufsteigend
        F_a = sum(1 for v in a if v <= x) / len(a)          # Anteil der Werte von a, die hoechstens x gross sind
        F_b = sum(1 for v in b if v <= x) / len(b)          # dasselbe fuer b
        abstand = abs(F_a - F_b)
        if zeigen:
            print(f"   x = {x:.2f}   F_ref = {F_a:.3f}   F_akt = {F_b:.3f}   Abstand = {abstand:.3f}")
        # TODO 3.2: zwei Zeilen: eine if-Zeile (abstand groesser als D?) und darunter eingerueckt D = abstand und wo = x
        raise NotImplementedError("TODO 3.2")
    return D, wo

# ---------- Teil A: das Folienbeispiel -----------------------------------------------------------------------------
ref_folie = [0.62, 0.81, 0.90, 0.95, 0.99]                  # 5 Konfidenzen Referenz
akt_folie = [0.41, 0.55, 0.70, 0.78, 0.88, 0.97]            # 6 Konfidenzen Produktion
D_folie, wo_folie = ks_von_hand(ref_folie, akt_folie, zeigen=True)
scipy_folie = ks_2samp(ref_folie, akt_folie)
print(f"von Hand: D = {D_folie:.3f} bei x = {wo_folie} | scipy: D = {scipy_folie.statistic:.3f}, p = {scipy_folie.pvalue:.3f}")

# ---------- Teil B: die eigenen Fenster ------------------------------------------------------------------------------
D_konf, wo_konf = ks_von_hand(konf_ref, konf_akt)
KS_KONF = ks_2samp(konf_ref, konf_akt)                      # die Bibliothek: Statistik und p-Wert
print(f"\nKonfidenz, Referenz gegen aktuell:  von Hand D = {D_konf:.3f} bei x = {wo_konf:.6f} | "
      f"scipy: D = {KS_KONF.statistic:.3f}, p = {KS_KONF.pvalue:.1e}")
print(f"   in Worten: {(konf_ref <= wo_konf).mean():.1%} der Referenz, aber {(konf_akt <= wo_konf).mean():.1%} des Driftstroms "
      f"haben eine Konfidenz von hoechstens {wo_konf:.6f}")

fig, ax = plt.subplots(figsize=(6.4, 3.3))
for werte, farbe, name in ((konf_ref, NAVY, "Referenz"), (konf_akt, ORANGE, "aktuell (Drift)")):
    ax.step(np.sort(werte), np.arange(1, len(werte) + 1) / len(werte), where="post", color=farbe, label=name)   # die Treppe
F_r, F_a = (konf_ref <= wo_konf).mean(), (konf_akt <= wo_konf).mean()
ax.plot([wo_konf, wo_konf], [F_r, F_a], color="black", lw=2); ax.text(wo_konf - 0.01, (F_r + F_a) / 2, f"D = {D_konf:.3f}", ha="right", va="center")
ax.set_xlabel("Konfidenz x"); ax.set_ylabel("F(x) = Anteil der Werte ≤ x"); ax.set_xlim(0.3, 1.005); ax.legend(frameon=False, loc="upper left")
ax.set_title("Zwei Treppen und ihr größter Abstand", color=NAVY, fontsize=11)
plt.tight_layout(); plt.savefig(RESULTS / "ks_konfidenz.png", dpi=150); plt.show()

check("3.2 Folienbeispiel von Hand: D = 0,467 bei x = 0,78", lambda: nah(D_folie, 0.4667, 0.0006) and wo_folie == 0.78)
check("3.2 Von Hand = scipy (Folienbeispiel und eigene Fenster)", lambda: nah(D_folie, scipy_folie.statistic, 1e-9) and nah(D_konf, KS_KONF.statistic, 1e-9))
check("3.2 Der p-Wert des Folienbeispiels ist gross (0,47): 11 Punkte beweisen nichts", lambda: nah(scipy_folie.pvalue, 0.474, 0.002))
check("3.2 Eigene Fenster: D zwischen 0,3 und 0,5 und p unter 0,001 (Referenzlauf: 0,395)", lambda: 0.3 < D_konf < 0.5 and KS_KONF.pvalue < 0.001)

*Ihre Antworten (`DENKEN`):*

1. Im Folienbeispiel ist $D$ = 0,467 **größer** als bei Ihren Fenstern – trotzdem ist der p-Wert dort 0,47 und hier
   praktisch 0. Warum?
2. An welcher Stelle $x$ liegt der größte Abstand bei Ihren Fenstern, und was bedeutet er in Worten
   („… % der Referenz, aber … % des Driftstroms haben eine Konfidenz von höchstens …“)?

### 3.3 · 512 Tests auf einmal – und warum man korrigieren muss

`PRINZIP` + `CODE` (1 Zeile) · Die Konfidenz ist **eine** Zahl je Bild. Das Embedding hat **512**. Der KS-Test kann nur
eine Spalte auf einmal – also testen wir jede Embedding-Dimension einzeln (Rabanser et al. 2019) und bekommen 512 p-Werte.

**Das Problem:** Ein Test mit Signifikanzniveau $\alpha = 0{,}05$ schlägt in 5 % der Fälle an, **obwohl sich nichts
geändert hat**. Bei 512 Tests erwarten wir also $512 \cdot 0{,}05 \approx 26$ Fehlalarme – irgendeine Dimension „driftet“
immer.

**Die Korrektur (Bonferroni):** Jeder einzelne Test muss strenger sein. Alarm nur, wenn der **kleinste** p-Wert unter

$$\frac{\alpha}{K} = \frac{0{,}05}{512} \approx 0{,}0001$$

liegt. Dann ist die Wahrscheinlichkeit für **mindestens einen** Fehlalarm über alle 512 Tests zusammen höchstens 5 %.

- **Teil A** zeigt das Problem: Die Referenz wird zufällig in zwei Hälften geteilt – beide stammen sicher aus derselben
  Verteilung – und Dimension für Dimension getestet (eine Schleife, 512 Aufrufe).
- **Teil B** ist die Funktion für den Alltag: `ks_2samp(..., axis=0)` rechnet alle 512 Tests in einem Aufruf.
  **Lücke** in `ks_bonferroni`: die Alarmregel – `alarm = bool(pw.min() < alpha / K)`

**Erwartung:** Zwei Hälften der Referenz: ohne Korrektur etwa 25–35 „driftende“ Dimensionen, mit Korrektur **0**.
Referenz gegen Driftstrom: **ALARM**, rund **40 %** der Dimensionen liegen unter der korrigierten Schwelle.

In [ ]:
# ---------- Teil A: Zwei Haelften der Referenz, Dimension fuer Dimension --------------------------------------------
rng = np.random.default_rng(0)                              # Zufallsgenerator mit festem Startwert (reproduzierbar)
reihenfolge = rng.permutation(len(E_ref))                   # die Zahlen 0 ... 500 in zufaelliger Reihenfolge
haelfte = len(reihenfolge) // 2
A, B = E_ref[reihenfolge[:haelfte]], E_ref[reihenfolge[haelfte:]]    # zwei Haelften derselben Stichprobe

p_werte = []
t0 = time.perf_counter()
for j in range(E_ref.shape[1]):                             # jede der 512 Dimensionen einzeln
    p_werte.append(ks_2samp(A[:, j], B[:, j]).pvalue)       # KS-Test auf Spalte j
T_SCHLEIFE = time.perf_counter() - t0
p_werte = np.array(p_werte)
K, alpha = len(p_werte), 0.05
print(f"Zwei Haelften der Referenz ({len(A)} gegen {len(B)} Bilder), {K} Tests:")
print(f"   erwartete Fehlalarme ohne Korrektur: {K} x {alpha} = {K * alpha:.1f}")
print(f"   p < 0,05        (ohne Korrektur): {(p_werte < alpha).sum():3d} Dimensionen 'driften' – obwohl sich nichts geaendert hat")
print(f"   p < 0,05 / {K}  (Bonferroni):     {(p_werte < alpha / K).sum():3d} Dimensionen   (Schwelle {alpha / K:.6f}, kleinster p-Wert {p_werte.min():.4f})")
HAELFTEN_OHNE = int((p_werte < alpha).sum())

# ---------- Teil B: die Funktion fuer den Alltag -----------------------------------------------------------------------
def ks_bonferroni(E_ref, E_akt, alpha=0.05):
    """KS-Test je Spalte, Bonferroni-Korrektur. Rueckgabe: (Alarm?, Anteil der Dimensionen unter alpha/K, alle p-Werte)."""
    pw = ks_2samp(E_ref, E_akt, axis=0).pvalue              # axis=0: ein Test je Spalte -> 512 p-Werte in einem Aufruf
    K = E_ref.shape[1]                                      # Anzahl der Tests
    # TODO 3.3: Alarm, wenn der kleinste p-Wert unter alpha / K liegt: alarm = bool(pw.min() kleiner alpha / K)
    raise NotImplementedError("TODO 3.3")
    return alarm, float((pw < alpha / K).mean()), pw

t0 = time.perf_counter()
_, _, pw_schnell = ks_bonferroni(A, B)
T_AXIS = time.perf_counter() - t0
print(f"\nRechenzeit fuer 512 Tests: Schleife {T_SCHLEIFE:.2f} s | ein Aufruf mit axis=0 {T_AXIS:.2f} s")
KS_ALARM, KS_ANTEIL, PW_EMB = ks_bonferroni(E_ref, E_akt)
print(f"Referenz gegen aktuell: {'ALARM' if KS_ALARM else 'kein Alarm'} – {KS_ANTEIL:.1%} der {K} Dimensionen unter alpha/K "
      f"(ohne Korrektur: {(PW_EMB < alpha).mean():.1%}), kleinster p-Wert {PW_EMB.min():.1e}")

fig, ax = plt.subplots(figsize=(6.4, 3.0))
kanten = np.linspace(0, 1, 21)
ax.hist(p_werte, bins=kanten, color=NAVY, alpha=0.85, label="Referenz gegen Referenz (zwei Hälften)")
ax.hist(PW_EMB, bins=kanten, color=ORANGE, alpha=0.75, label="Referenz gegen Driftstrom")
ax.set_xlabel("p-Wert des KS-Tests je Embedding-Dimension"); ax.set_ylabel("Anzahl Dimensionen"); ax.legend(frameon=False)
ax.set_title("512 p-Werte: ohne Drift gleichmäßig verteilt, mit Drift bei 0 gestapelt", color=NAVY, fontsize=11)
plt.tight_layout(); plt.savefig(RESULTS / "pwerte_embedding.png", dpi=150); plt.show()

check("3.3 Schleife und axis=0 liefern dieselben 512 p-Werte", lambda: np.allclose(p_werte, pw_schnell))
check("3.3 Zwei Haelften: ohne Korrektur 'driften' einige Dimensionen (10 bis 50), mit Bonferroni keine",
      lambda: 10 <= HAELFTEN_OHNE <= 50 and (p_werte < alpha / K).sum() == 0)
check("3.3 Referenz gegen Driftstrom: Alarm, 25 % bis 55 % der Dimensionen (Referenzlauf: 39,5 %)", lambda: KS_ALARM and 0.25 < KS_ANTEIL < 0.55)

*Ihre Antworten (`DENKEN`):*

1. Im Histogramm sind die p-Werte der beiden Hälften ungefähr gleichmäßig zwischen 0 und 1 verteilt. Warum muss das so
   sein, wenn sich nichts geändert hat?
2. Bonferroni ist streng. Welchen Preis zahlt man dafür – was könnte der Detektor übersehen?

### 3.4 · Gegenprobe: Referenz gegen Referenz

`CODE` (1 Zeile) · Ein Detektor, der beim Driftstrom anschlägt, ist erst die halbe Miete. Die andere Hälfte: **Bleibt er
still, wenn sich nichts geändert hat?** Wir teilen die Referenz **20-mal** zufällig in zwei Hälften und lassen beide
Detektoren laufen. Jeder Alarm ist ein Fehlalarm.

**Lücke:** die beiden Hälften bilden – `a, b = reihenfolge[:haelfte], reihenfolge[haelfte:]`

**Erwartung:** Höchstens ein bis zwei Fehlalarme des KS-Tests mit Bonferroni – die Korrektur verspricht „höchstens 5 %“,
nicht „nie“. Der PSI bleibt in allen 20 Durchgängen unter 0,1.

In [ ]:
zeilen = []
for durchgang in range(20):                                 # 20 verschiedene zufaellige Teilungen
    rng = np.random.default_rng(durchgang)                  # anderer Startwert -> andere Teilung
    reihenfolge = rng.permutation(len(E_ref))
    haelfte = len(reihenfolge) // 2
    # TODO 3.4: die beiden Haelften: a, b = reihenfolge[:haelfte], reihenfolge[haelfte:]
    raise NotImplementedError("TODO 3.4")
    alarm, anteil, pw = ks_bonferroni(E_ref[a], E_ref[b])
    zeilen.append({"Durchgang": durchgang, "PSI Konfidenz": psi(konf_ref[a], konf_ref[b]),
                   "kleinster p-Wert": pw.min(), "KS-Alarm": alarm})
GEGENPROBE = pd.DataFrame(zeilen).set_index("Durchgang")
display(GEGENPROBE.head(5).round(4))
print(f"20 Gegenproben: {int(GEGENPROBE['KS-Alarm'].sum())} KS-Fehlalarme | groesster PSI {GEGENPROBE['PSI Konfidenz'].max():.3f} "
      f"(Driftstrom: {PSI_KONF:.3f}) | kleinster p-Wert ueberhaupt {GEGENPROBE['kleinster p-Wert'].min():.5f} (Schwelle {0.05 / 512:.6f})")
check("3.4 Hoechstens zwei KS-Fehlalarme in 20 Gegenproben", lambda: GEGENPROBE["KS-Alarm"].sum() <= 2)
check("3.4 Der PSI bleibt in allen Gegenproben unter 0,1", lambda: GEGENPROBE["PSI Konfidenz"].max() < 0.1)
check("3.4 Der PSI des Driftstroms ist mindestens fuenfmal so gross wie der groesste der Gegenproben",
      lambda: PSI_KONF > 5 * GEGENPROBE["PSI Konfidenz"].max())

*Ihre Antworten (`DENKEN`):*

1. Der PSI zweier Hälften derselben Stichprobe ist nicht 0. Woher kommt der Rest?
2. Was sagt Ihnen die Gegenprobe über die PSI-Schwelle 0,25 – ist sie für **dieses** Modell und **diese** Fenstergröße
   eher vorsichtig oder eher knapp gewählt?
3. Gab es bei Ihnen einen KS-Fehlalarm? Ist der Detektor dann kaputt?

## Aufgabe 4 · Evidently und Schwellwert

**In der Vorlesung: Live 2, Schritte 4 und 5 · ≈ 25 Min.** · Folien „Vom Messwert zum Auslöser“ und Anhang B
„Code: Evidently-Bericht“

Sie haben den Detektor von Hand gebaut. In der Praxis nimmt man dafür meist ein fertiges Werkzeug – und muss dann wissen,
**welche Regel** es anwendet. Danach wird aus Kennzahlen eine Entscheidung.

| Teil | Inhalt | Art | Zeit |
|---|---|---|---|
| 4.1 | Evidently-Bericht erzeugen – und seine Regel von Hand nachrechnen | `CODE` (2 Zeilen) + `PRINZIP` | 10 Min. |
| 4.2 | Zweistufige Alarmregel: erst ohne, dann mit Labels | `CODE` (2 Zeilen) + `PRINZIP` | 10 Min. |
| 4.3 | Schwellwert in fünf Sätzen begründen | `DENKEN` · **Abgabe** | 5 Min. |

### 4.1 · Evidently-Bericht erzeugen

`CODE` (2 Zeilen) · Evidently (im Kurs: Version **0.4.40**) bekommt zwei Tabellen mit denselben Spalten – Referenz und
aktuell – und erzeugt einen HTML-Bericht. Unsere Tabellen haben **11 Spalten**: die Konfidenz und, zur Übersicht, die
ersten zehn Embedding-Dimensionen `e0` … `e9`.

**Lücke:**

- `bericht = Report(metrics=[DataDriftPreset()])`
- `bericht.run(reference_data=tab_ref, current_data=tab_akt)`

Der Bericht landet in `lab08/results/drift.html` – im Jupyter-Dateibrowser öffnen (oder im Dateimanager doppelklicken).

**Erwartung:** Evidently meldet **Datensatz-Drift**; der Anteil driftender Spalten liegt über 50 %.

In [ ]:
abschnitt("Aufgabe 4")
from evidently.report import Report                         # API von evidently 0.4.x (ab 0.7 heisst vieles anders)
from evidently.metric_preset import DataDriftPreset

def tabelle(konf, E, n_dim=10):
    """Tabelle fuer Evidently: Spalte 'konfidenz' und die ersten n_dim Embedding-Dimensionen e0, e1, ..."""
    return pd.DataFrame({"konfidenz": konf, **{f"e{j}": E[:, j] for j in range(n_dim)}})

tab_ref, tab_akt = tabelle(konf_ref, E_ref), tabelle(konf_akt, E_akt)
# TODO 4.1: bericht = Report(metrics=[DataDriftPreset()]) und bericht.run(reference_data=tab_ref, current_data=tab_akt)
raise NotImplementedError("TODO 4.1")
bericht.save_html(str(RESULTS / "drift.html"))              # Bericht zum Ansehen im Browser
ergebnis = bericht.as_dict()["metrics"]                     # dasselbe Ergebnis als Python-Datenstruktur
gesamt = ergebnis[0]["result"]                              # 1. Metrik: das Gesamturteil
je_spalte = pd.DataFrame(ergebnis[1]["result"]["drift_by_columns"]).T[["stattest_name", "stattest_threshold", "drift_score", "drift_detected"]]
display(je_spalte)
print(f"Datensatz-Drift: {gesamt['dataset_drift']} | driftende Spalten: {gesamt['number_of_drifted_columns']} von {gesamt['number_of_columns']}"
      f" = {gesamt['share_of_drifted_columns']:.0%} | Regel: Drift ab einem Anteil von {gesamt['drift_share']:.0%}")
print(f"Bericht: lab08/results/drift.html ({(RESULTS / 'drift.html').stat().st_size / 1e6:.1f} MB)")
check("4.1 Bericht erzeugt und gespeichert", lambda: (RESULTS / "drift.html").stat().st_size > 100_000)
check("4.1 Evidently meldet Datensatz-Drift", lambda: gesamt["dataset_drift"] is True)

#### 4.1 `PRINZIP` · Evidentlys Regel von Hand

`PRINZIP` · In der Tabelle oben steht, was Evidently getan hat: je Spalte ein **KS-Test** (`stattest_name`; das wählt
Evidently bei Zahlenspalten mit bis zu 1 000 Zeilen selbst), Drift bei **p < 0,05** (`stattest_threshold`),
**ohne** Korrektur für viele Tests. Datensatz-Drift meldet es, wenn **mindestens die Hälfte** der Spalten driftet.

Die Zelle rechnet genau das in einer Schleife nach – und wendet dieselbe Regel danach auf die **Gegenprobe** an (zwei
Hälften der Referenz), einmal mit unseren 11 Spalten und einmal mit allen 512 Embedding-Dimensionen.

**Erwartung:** Von Hand dieselben p-Werte und dasselbe Urteil wie Evidently. In der Gegenprobe driften einzelne Spalten
„zufällig“, der Datensatz als Ganzes aber nicht.

In [ ]:
def evidently_regel(tab_a, tab_b, alpha=0.05, anteil_schwelle=0.5):
    """Evidentlys Standardregel fuer Zahlenspalten bis 1000 Zeilen, von Hand: KS-Test je Spalte, Drift ab der Haelfte."""
    driftend = []
    for spalte in tab_a.columns:                            # eine Spalte nach der anderen
        p_wert = ks_2samp(tab_a[spalte], tab_b[spalte]).pvalue
        if p_wert < alpha:                                  # ohne Korrektur fuer viele Tests
            driftend.append(spalte)
    anteil = len(driftend) / len(tab_a.columns)
    return anteil >= anteil_schwelle, anteil, driftend

drift_hand, anteil_hand, spalten_hand = evidently_regel(tab_ref, tab_akt)
print(f"Von Hand:   Datensatz-Drift {drift_hand} | {len(spalten_hand)} von {len(tab_ref.columns)} Spalten = {anteil_hand:.0%}: {spalten_hand}")
print(f"Evidently:  Datensatz-Drift {gesamt['dataset_drift']} | {gesamt['number_of_drifted_columns']} von {gesamt['number_of_columns']} Spalten")
p_hand_e0 = ks_2samp(tab_ref["e0"], tab_akt["e0"]).pvalue
print(f"p-Wert Spalte e0: von Hand {p_hand_e0:.6f} | Evidently {je_spalte.loc['e0', 'drift_score']:.6f}")

# ---------- Gegenprobe mit Evidentlys Regel ------------------------------------------------------------------------
rng = np.random.default_rng(0); reihenfolge = rng.permutation(len(E_ref)); haelfte = len(reihenfolge) // 2
a, b = reihenfolge[:haelfte], reihenfolge[haelfte:]
g11 = evidently_regel(tabelle(konf_ref[a], E_ref[a]), tabelle(konf_ref[b], E_ref[b]))
g512 = evidently_regel(pd.DataFrame(E_ref[a]), pd.DataFrame(E_ref[b]))
print(f"\nGegenprobe, 11 Spalten:   Datensatz-Drift {g11[0]} | driftende Spalten: {len(g11[2])} {g11[2]}")
print(f"Gegenprobe, 512 Spalten:  Datensatz-Drift {g512[0]} | driftende Spalten: {len(g512[2])} = {g512[1]:.1%}")
alle512 = evidently_regel(pd.DataFrame(E_ref), pd.DataFrame(E_akt))
print(f"Driftstrom, 512 Spalten:  Datensatz-Drift {alle512[0]} | driftende Spalten: {len(alle512[2])} = {alle512[1]:.1%}")

check("4.1 PRINZIP: von Hand dasselbe Urteil und dieselbe Zahl driftender Spalten wie Evidently",
      lambda: drift_hand == gesamt["dataset_drift"] and len(spalten_hand) == gesamt["number_of_drifted_columns"])
check("4.1 PRINZIP: p-Wert von Hand = drift_score von Evidently (Spalte e0)", lambda: nah(p_hand_e0, je_spalte.loc["e0", "drift_score"], 1e-9))
check("4.1 PRINZIP: Gegenprobe - kein Datensatz-Drift (weder mit 11 noch mit 512 Spalten)", lambda: not g11[0] and not g512[0])

*Ihre Antworten (`DENKEN`):*

1. Evidently korrigiert nicht für viele Tests. Warum gibt es in der Gegenprobe trotzdem keinen Datensatz-Alarm?
2. Zwei Regeln, ein Ziel: „kleinster p-Wert unter α/K“ (Bonferroni, Aufgabe 3.3) und „mindestens die Hälfte der Spalten
   mit p < 0,05“ (Evidently). Welche schlägt bei einer Störung an, die nur **wenige** Dimensionen stark trifft? Welche
   bei einer, die **alle** Dimensionen ein wenig trifft?

### 4.2 · Zweistufige Alarmregel

`CODE` (2 Zeilen) + `PRINZIP` · Folie „Vom Messwert zum Auslöser“. Ein Drift-Alarm heißt: **Die Verteilung hat sich
geändert.** Er heißt nicht: Das Modell ist schlechter geworden. Deshalb zwei Stufen:

- **Stufe 1 – ohne Labels, läuft ständig:** Alarm, wenn der **PSI der Konfidenz über 0,25** liegt **oder** der
  **KS-Test mit Bonferroni** auf den Embeddings anschlägt. Folge: Ein Prüfer labelt eine Stichprobe.
  **Lücke 4.2a** in `stufe1`: `return psi_wert > PSI_SCHWELLE or ks_alarm`
- **Stufe 2 – mit Labels, nur nach einem Alarm:** Der Prüfer labelt **100** zufällig gezogene Bilder des aktuellen
  Fensters. Sind **weniger als 95** davon richtig klassifiziert (Genauigkeit unter 0,95 – die Referenz liegt bei 0,984),
  wird das Re-Training ausgelöst. **Lücke 4.2b:** `ausloeser = richtig < MIN_RICHTIG`

**Warum 100 und 95?** Das ist die Abwägung dieses Abschnitts, und die Zelle rechnet sie in einer Simulation nach
(`PRINZIP`): Wir ziehen 2 000-mal eine Stichprobe von 100 Bildern – einmal aus der Referenz (jeder Auslöser wäre ein
**Fehlalarm**) und einmal aus dem Driftstrom (jeder Nicht-Auslöser wäre ein **verpasster Drift**).

Jetzt kommen die beiseitegelegten Labels `y_ref` und `y_akt` zum Einsatz: Sie spielen den Prüfer.

**Erwartung:** Stufe 1: ALARM. Fehlalarm-Wahrscheinlichkeit von Stufe 2 unter 1 %; der Drift wird in etwa vier von
fünf Stichproben erkannt. (Die Simulation zieht aus nur 501 Referenzbildern, von denen 8 falsch klassifiziert sind –
sechs davon in einer Stichprobe zu erwischen ist fast unmöglich. Die Formel daneben rechnet mit einem beliebig langen
Strom und ist deshalb etwas vorsichtiger.)

In [ ]:
PSI_SCHWELLE = 0.25          # Stufe 1: Faustschwelle "handeln"
N_STICHPROBE = 100           # Stufe 2: so viele Bilder labelt der Pruefer
MIN_RICHTIG  = 95            # Stufe 2: weniger als 95 von 100 richtig -> Re-Training-Ausloeser

def stufe1(psi_wert, ks_alarm):
    """Stufe 1 (ohne Labels): True = Alarm."""
    # TODO 4.2a: return psi_wert groesser PSI_SCHWELLE or ks_alarm
    raise NotImplementedError("TODO 4.2a")

ALARM1 = stufe1(PSI_KONF, KS_ALARM)
print(f"Stufe 1:  PSI {PSI_KONF:.3f} (Schwelle {PSI_SCHWELLE}) | KS mit Bonferroni: {'Alarm' if KS_ALARM else 'kein Alarm'}"
      f"  ->  {'ALARM: Stichprobe labeln lassen' if ALARM1 else 'kein Alarm'}")

# ---------- Der Pruefer: richtig oder falsch je Bild (dafuer brauchen wir die Labels) -------------------------------
richtig_ref = (P_ref.argmax(axis=1) == y_ref)               # je Bild True/False: vorhergesagte Klasse = richtige Klasse?
richtig_akt = (P_akt.argmax(axis=1) == y_akt)
ACC_REF, ACC_AKT = richtig_ref.mean(), richtig_akt.mean()
print(f"\nGenauigkeit ueber alle {len(y_ref)} Bilder (das weiss im Betrieb niemand):  Referenz {ACC_REF:.4f} | Driftstrom {ACC_AKT:.4f}")

# ---------- PRINZIP: Wie oft loest Stufe 2 aus? Simulation mit 2000 Stichproben ----------------------------------------
def ausloese_quote(richtig, n=N_STICHPROBE, minimum=MIN_RICHTIG, wiederholungen=2000, seed=1):
    rng = np.random.default_rng(seed)
    ausgeloest = 0
    for _ in range(wiederholungen):                         # immer wieder: n Bilder zufaellig ziehen (ohne Zuruecklegen)
        stichprobe = rng.choice(richtig, size=n, replace=False)
        if stichprobe.sum() < minimum:                      # weniger als "minimum" richtige -> Ausloeser
            ausgeloest += 1
    return ausgeloest / wiederholungen

FEHLALARM = ausloese_quote(richtig_ref)                     # Referenz: jeder Ausloeser ist ein Fehlalarm
ERKANNT = ausloese_quote(richtig_akt)                       # Driftstrom: jeder Ausloeser ist ein Treffer
print(f"Simulation, Stichprobe {N_STICHPROBE}, Ausloeser bei weniger als {MIN_RICHTIG} richtigen:")
print(f"   Fehlalarm (Referenz):       {FEHLALARM:.1%} der Stichproben")
print(f"   Drift erkannt (Driftstrom): {ERKANNT:.0%} der Stichproben  ->  verpasst: {1 - ERKANNT:.0%}")
print(f"   zum Vergleich die Formel (Binomialverteilung, Genauigkeit 0,984, beliebig langer Strom): Fehlalarm {binom.cdf(MIN_RICHTIG - 1, N_STICHPROBE, 0.984):.1%}")

# ---------- Die Entscheidung fuer DIESES Fenster: eine Stichprobe ziehen ---------------------------------------------
rng = np.random.default_rng(8)
gezogen = rng.choice(len(richtig_akt), size=N_STICHPROBE, replace=False)     # 100 zufaellige Zeilennummern
richtig = int(richtig_akt[gezogen].sum())                   # so viele hat das Modell davon richtig
# TODO 4.2b: ausloeser = richtig kleiner MIN_RICHTIG
raise NotImplementedError("TODO 4.2b")
print(f"\nStufe 2:  Pruefer labelt {N_STICHPROBE} Bilder -> {richtig} richtig (Genauigkeit {richtig / N_STICHPROBE:.2f})"
      f"  ->  {'RE-TRAINING-AUSLOESER' if ALARM1 and ausloeser else 'kein Ausloeser'}")

# ---------- Die Abwaegung als Tabelle: andere Stichprobengroessen, gleiche Schwelle 0,95 --------------------------------
ABWAEGUNG = pd.DataFrame([{"Stichprobe n": n, "Ausloeser bei weniger als": math.ceil(0.95 * n),
                           "Fehlalarm": ausloese_quote(richtig_ref, n, math.ceil(0.95 * n)),
                           "Drift erkannt": ausloese_quote(richtig_akt, n, math.ceil(0.95 * n))} for n in (25, 50, 100, 200)]).set_index("Stichprobe n")
display(ABWAEGUNG.style.format({"Fehlalarm": "{:.1%}", "Drift erkannt": "{:.0%}"}))

check("4.2 Stufe 1 schlaegt beim Driftstrom an", lambda: ALARM1 is True or ALARM1 == True)
check("4.2 stufe1(): kein Alarm bei PSI 0,05 ohne KS-Alarm, Alarm bei PSI 0,30, Alarm bei KS-Alarm",
      lambda: not stufe1(0.05, False) and stufe1(0.30, False) and stufe1(0.05, True))
check("4.2 Genauigkeit der Referenz 0,984; im Driftstrom mindestens 3 Punkte niedriger", lambda: nah(ACC_REF, 0.984, 0.004) and ACC_AKT < ACC_REF - 0.03)
check("4.2 Stufe 2: Fehlalarm unter 2 %, Drift in mehr als der Haelfte der Stichproben erkannt", lambda: FEHLALARM < 0.02 and ERKANNT > 0.5)
check("4.2 Mit 200 gelabelten Bildern wird der Drift oefter erkannt als mit 25", lambda: ABWAEGUNG.loc[200, "Drift erkannt"] > ABWAEGUNG.loc[25, "Drift erkannt"])

### 4.3 · Schwellwert in fünf Sätzen begründen

`DENKEN` · **Abgabe.** Schreiben Sie fünf Sätze – für jemanden, der den Dienst betreibt und wissen will, warum nachts
der Alarm klingelt:

1. **Was** überwachen Sie in Stufe 1 und gegen welche Referenz?
2. **Welche Schwellen** gelten in Stufe 1, und was haben Sie getan, um Fehlalarme auszuschließen?
3. **Was passiert** bei einem Alarm der Stufe 1 – und warum wird nicht sofort nachtrainiert?
4. **Welche Schwelle** gilt in Stufe 2, und welche Fehlalarm- und Erkennungsquote hat sie laut Ihrer Simulation?
5. **Was würden Sie ändern**, wenn ein verpasster Drift sehr teuer wäre (z. B. Sichtprüfung sicherheitsrelevanter Teile)?

*Ihre Begründung:*

## Zusatz (freiwillig) · Z1 Schleichender Drift über sechs „Tage“

`AUSFÜHREN` + `DENKEN` · Bisher kam die Störung auf einen Schlag. In der Praxis altert eine Lampe, verstellt sich eine
Kamera langsam. Wir simulieren sechs Tage: An Tag 0 sind die Bilder unverändert, an Tag 5 so gestört wie im Driftstrom,
dazwischen wächst die Stärke gleichmäßig (Helligkeit 1,0 → 1,6 · σ 0 → 1,5 · JPEG-Qualität 100 → 20).

Die Zelle schickt die Bilder diesmal **direkt** durch das Modell im Notebook (ohne Dienst, in Paketen zu 64 – das ist
schneller) und berechnet je Tag: Genauigkeit, PSI, Anteil driftender Dimensionen, Urteil von Stufe 1 und – wenn wir alle
501 Labels hätten – ob die Genauigkeit unter 0,95 liegt.

**Vorher schätzen:** An welchem Tag schlägt Stufe 1 zum ersten Mal an? Und an welchem Tag liegt die Genauigkeit zum
ersten Mal unter 0,95?

In [ ]:
%%nur_mit_dienst
abschnitt("Zusatz")
def strom_direkt(staerke):
    """Alle Validierungsbilder mit Stoerstaerke 0 ... 1 direkt durch das Modell: (P, E)."""
    bilder = [Image.open(BILDER / pfad).convert("RGB") for pfad, _ in VAL]
    if staerke > 0:
        bilder = [stoere(b, helligkeit=1 + 0.6 * staerke, sigma=1.5 * staerke, qualitaet=round(100 - 80 * staerke)) for b in bilder]
    P, E = [], []
    with torch.no_grad():
        for i in range(0, len(bilder), 64):                 # in Paketen zu 64 Bildern
            x = torch.stack([tf(b) for b in bilder[i:i + 64]])
            for name, schicht in modell.named_children():   # wie in 1.1 PRINZIP: bis vor die letzte Schicht
                if name == "fc":
                    break
                x = schicht(x)
            emb = x.flatten(1)
            P.append(torch.softmax(modell.fc(emb), dim=1).numpy()); E.append(emb.numpy())
    return np.concatenate(P), np.concatenate(E)

y_val = np.array([int(label) for _, label in VAL])
tage = []
for tag in range(6):
    P_t, E_t = strom_direkt(tag / 5)
    alarm_ks, anteil_ks, _ = ks_bonferroni(E_ref, E_t)
    psi_t = psi(konf_ref, P_t.max(axis=1))
    tage.append({"Tag": tag, "Stärke": tag / 5, "Genauigkeit": (P_t.argmax(axis=1) == y_val).mean(), "PSI Konfidenz": psi_t,
                 "KS-Alarm": alarm_ks, "Anteil Dim. unter α/K": anteil_ks, "Stufe 1": stufe1(psi_t, alarm_ks)})
TAGE = pd.DataFrame(tage).set_index("Tag")
TAGE["Genauigkeit unter 0,95"] = TAGE["Genauigkeit"] < 0.95
display(TAGE.round(3))

fig, ax = plt.subplots(figsize=(6.6, 3.2)); ax2 = ax.twinx()
ax.plot(TAGE.index, TAGE["Genauigkeit"], "o-", color=NAVY, label="Genauigkeit"); ax.axhline(0.95, color=NAVY, ls=":", lw=1)
ax2.plot(TAGE.index, TAGE["PSI Konfidenz"], "s-", color=ORANGE, label="PSI Konfidenz"); ax2.axhline(0.25, color=ORANGE, ls=":", lw=1)
ax.set_xlabel("Tag"); ax.set_ylabel("Genauigkeit", color=NAVY); ax2.set_ylabel("PSI", color=ORANGE); ax2.spines["right"].set_visible(True)
erster = TAGE.index[TAGE["Stufe 1"]].min()
ax.axvline(erster, color=GRAU, ls="--"); ax.text(erster + 0.05, ax.get_ylim()[0] + 0.002, "Stufe 1 schlägt an", color=GRAU, fontsize=9)
ax.set_title("Schleichender Drift: Der Detektor ist früher als der Schaden", color=NAVY, fontsize=11)
plt.tight_layout(); plt.savefig(RESULTS / "schleichender_drift.png", dpi=150); plt.show()

check("Z1 Tag 0 (keine Stoerung): kein Alarm, PSI = 0", lambda: not TAGE.loc[0, "Stufe 1"] and nah(TAGE.loc[0, "PSI Konfidenz"], 0, 0.01))
check("Z1 Tag 5 entspricht dem Driftstrom (PSI wie in 3.1, Abweichung hoechstens 0,05)", lambda: nah(TAGE.loc[5, "PSI Konfidenz"], PSI_KONF, 0.05))
check("Z1 Stufe 1 schlaegt frueher an (oder am selben Tag), als die Genauigkeit unter 0,95 faellt",
      lambda: TAGE.index[TAGE["Stufe 1"]].min() <= (TAGE.index[TAGE["Genauigkeit unter 0,95"]].min() if TAGE["Genauigkeit unter 0,95"].any() else 99))

*Ihre Antworten (`DENKEN`):*

1. An welchem Tag schlägt welcher Teil von Stufe 1 an – KS mit Bonferroni, PSI? Wie hoch ist an diesem Tag die Genauigkeit?
2. „Der Drift-Test ist signifikant, also ist das Modell schlecht.“ Widerlegen Sie den Satz mit einer Zeile der Tabelle.
3. Was hieße es für den Prüfer, wenn Stufe 1 **allein** das Re-Training auslösen würde?

## Zusatz (freiwillig) · Z2 Label Shift: andere Mischung, gleiche Bilder

`AUSFÜHREN` + `DENKEN` · Folie „Drei Stellen, an denen sich etwas verschiebt“. Bisher änderte sich, wie die Bilder
**aussehen** (Covariate Shift). Jetzt ändert sich, **wie oft jede Klasse vorkommt**: Stellen Sie sich vor, an der Anlage
laufen diese Woche fast nur Golfbälle und Fallschirme durch. Jedes einzelne Bild ist ein ganz normales Bild – nur die
Mischung ist eine andere. Das ist *Label Shift*: $P(Y)$ ändert sich, $P(X \mid Y)$ bleibt.

Die Zelle braucht keinen Dienst: Sie teilt das Referenzfenster in zwei Hälften. Die erste bleibt Referenz. Aus der zweiten
nimmt sie **alle** Golfbälle und Fallschirme, aber nur **jedes fünfte** Bild der anderen drei Klassen.

**Vorher schätzen:** Schlägt Stufe 1 an? Und ist das Modell auf diesem Fenster schlechter geworden?

In [ ]:
rng = np.random.default_rng(0); reihenfolge = rng.permutation(len(y_ref)); haelfte = len(reihenfolge) // 2
ref_idx, pool = reihenfolge[:haelfte], reihenfolge[haelfte:]            # erste Haelfte = Referenz, zweite = Vorrat
haeufig = [KLASSEN.index("Golfball"), KLASSEN.index("Fallschirm")]
behalten = [i for nr, i in enumerate(pool) if y_ref[i] in haeufig or nr % 5 == 0]      # seltene Klassen: nur jedes fuenfte Bild
behalten = np.array(behalten)

def klassen_anteile(P):
    """Anteil jeder Klasse unter den VORHERSAGEN (dafuer braucht man keine Labels)."""
    return np.bincount(P.argmax(axis=1), minlength=len(KLASSEN)) / len(P)

v_ref, v_akt = klassen_anteile(P_ref[ref_idx]), klassen_anteile(P_ref[behalten])
display(pd.DataFrame({"Referenz": v_ref, "aktuell": v_akt}, index=KLASSEN).style.format("{:.1%}"))
PSI_KLASSEN = psi_aus_anteilen(v_ref, v_akt)                # dieselbe PSI-Formel, Klassen = die fuenf vorhergesagten Klassen
psi_k = psi(konf_ref[ref_idx], konf_ref[behalten])
alarm_ls, anteil_ls, _ = ks_bonferroni(E_ref[ref_idx], E_ref[behalten])
acc_ls_ref = (P_ref[ref_idx].argmax(axis=1) == y_ref[ref_idx]).mean()
acc_ls_akt = (P_ref[behalten].argmax(axis=1) == y_ref[behalten]).mean()
print(f"Fenster: Referenz {len(ref_idx)} Bilder, aktuell {len(behalten)} Bilder")
print(f"PSI der vorhergesagten Klassen: {PSI_KLASSEN:.3f} | PSI der Konfidenz: {psi_k:.3f}")
print(f"KS mit Bonferroni auf den Embeddings: {'ALARM' if alarm_ls else 'kein Alarm'} ({anteil_ls:.1%} der Dimensionen)")
print(f"Stufe 1: {'ALARM' if stufe1(psi_k, alarm_ls) else 'kein Alarm'}")
print(f"Genauigkeit: Referenz {acc_ls_ref:.3f} | aktuell {acc_ls_akt:.3f}")
check("Z2 Die Mischung der vorhergesagten Klassen hat sich deutlich verschoben (PSI ueber 0,25)", lambda: PSI_KLASSEN > 0.25)
check("Z2 Die Embeddings schlagen Alarm", lambda: alarm_ls)
check("Z2 Die Genauigkeit ist nicht eingebrochen (hoechstens 3 Punkte unter der Referenz)", lambda: acc_ls_akt > acc_ls_ref - 0.03)

*Ihre Antworten (`DENKEN`):*

1. Welcher Detektor hat angeschlagen, welcher nicht? Warum sieht der PSI der **Konfidenz** diese Verschiebung kaum?
2. Sollte hier nachtrainiert werden? Was sagt Stufe 2?
3. Woran erkennen Sie – ohne Labels –, dass es sich eher um Label Shift als um Covariate Shift handelt?

## Abgabe, Laufzeiten und Prüfungen

`AUSFÜHREN` · Die Zelle beendet den Dienst und packt `lab08/app.py` und `lab08/senden.py` in ein ZIP neben diesem
Notebook (`lab08/results/abgabe_lab08.zip`). Abzugeben sind **dieses Notebook mit Ausgaben und Ihren Antworten** –
vor allem 4.3 – und das ZIP.

**Aufräumen:** Das Protokoll `traeger-pipeline/results/inferenz_log.jsonl` wächst mit jedem gesendeten Strom um einige
Megabyte. Sie können die Datei jederzeit löschen; der Dienst legt sie neu an.

In [ ]:
abschnitt("Abschluss")
stoppe()                                                      # alle Hintergrundprogramme beenden
zip_pfad = RESULTS / "abgabe_lab08.zip"
namen = [n for n in ("lab08/app.py", "lab08/senden.py") if (ARBEIT / n).exists()]
with zipfile.ZipFile(zip_pfad, "w", zipfile.ZIP_DEFLATED) as z:
    for n in namen:
        z.write(ARBEIT / n, arcname=n)
print(f"results/{zip_pfad.name}: {len(namen)} Dateien" + (" – " + ", ".join(namen) if namen else " (ohne Dienst gibt es keine Dateien abzugeben)"))

display(pd.DataFrame({"Rechenzeit (s)": RUNTIME}).round(1))
if LAUFZEITEN:
    print("Befehle:")
    display(pd.Series(LAUFZEITEN, name="s").round(1))
print(f"Pruefungen: {sum(CHECKS.values())} ✅ / {sum(not v for v in CHECKS.values())} ❌")
for name, ok in CHECKS.items():
    if not ok:
        print("  ❌", name)

## Weiterdenken (für das Capstone)

- **Monitoring-Konzept:** Tragen Sie für Ihr Projekt die fünf Spalten der Folie „Das Monitoring-Konzept auf einer Seite“
  ein – *was* · *womit* · *gegen welche Referenz* · *welcher Schwellwert* · *was passiert beim Alarm*. Mindestens je eine
  Zeile ohne Labels, mit Labels und für den Betrieb. Dieses Lab ist das ausgefüllte Beispiel für das Trägermodell.
- **Welche Stellvertreter haben Sie?** Bei Bildern: Embeddings und Konfidenz. Bei Tabellen: die Merkmale selbst. Bei
  Texten: Embeddings, Textlänge, Anteil unbekannter Wörter. Bei Zeitreihen: die Residuen der Vorhersage.
- **Betriebsmetriken:** Latenz und Fehlerrate fehlen in diesem Lab. Die Anhangfolien „Prometheus und Grafana“ und
  „Code: Metriken am FastAPI-Dienst“ zeigen, wie der Dienst sie mit wenigen Zeilen selbst zählt.
- **Nach dem Alarm:** Wie kommt ein neu trainiertes Modell sicher in Betrieb – und trainiert man auf Zuruf oder nach
  Kalender? Das beantwortet der Gruppenvortrag „Testen in Produktion & Re-Training“.